# Guardian Layer research: preprocessing, baseline model, evaluation, optimization

The experiments behind the preprocessing pipeline (#9), the anomaly baseline model
(#10), its evaluation (#12) and its optimization (#64). This notebook replaces the
scripts that used to live in `research/`; the written findings stay in the
`results.md` files next to it.

How to run: open it from inside the repository (any folder; the repository root is
found from `pyproject.toml`) and run the cells in order. The long experiments
use up to 8 worker processes (about 0.7 GB each), so the full comparison still takes
a while. The datasets are read from the shared `datasets.root` folder
(`config/datasets.yaml`), and each source's readings are cached once in
`~/.cache/guardian_layer/readings` (or `$GUARDIAN_DATA_DIR/readings`). Change
`METHOD` below to try another decoupling method; `MODEL_DIR` is where the trained
baseline model is written (git-ignored).

## Setup

In [ ]:
import cProfile
import dataclasses
import hashlib
import json
import logging
import logging.handlers
import math
import os
import pickle
import platform
import pstats
import sys
from collections.abc import Callable
from datetime import UTC, datetime
from functools import partial
from itertools import pairwise
from pathlib import Path
from time import perf_counter
from typing import Any

import numpy as np
import numpy.typing as npt
from joblib import Parallel, delayed
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from guardian_layer.ai.dataset_factory import DatasetFactory
from guardian_layer.ai.datasets.windows import (
    LabelledWindows,
    Process,
    process_window,
)
from guardian_layer.ai.inference import (
    AnomalyDetector,
    BaselineModel,
    LongAnomalyDetector,
)
from guardian_layer.ai.inference.detector import model_verdicts
from guardian_layer.pipelines.decompose import Decompose
from guardian_layer.pipelines.denoise import kalman as kalman_clean
from guardian_layer.pipelines.long_preprocess import (
    LongPreProcess,
    process_long_window,
)
from guardian_layer.pipelines.preprocess import PreProcess
from guardian_layer.pipelines.static_check import StaticCheck
from guardian_layer.utils.config import (
    KALMAN_MODES,
    KALMAN_NOISES,
    KalmanSettings,
    LongWindowSettings,
    PreprocessSettings,
    get_settings,
)
from guardian_layer.utils.dataset_config import DatasetSettings, get_dataset_settings
from guardian_layer.utils.schema import (
    Anomalies,
    BoolArray,
    CheckResult,
    CleanContext,
    FloatArray,
    IntArray,
    LabeledReading,
    ReadingColumns,
    Sensor,
    SensorSummary,
    SourceReadings,
    WindowDataset,
)

In [ ]:
# the repository root: the nearest parent folder holding pyproject.toml
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()),
    None,
)
assert ROOT is not None, "open this notebook from inside the repository"
sys.path.insert(0, str(ROOT))  # mocks/ is not an installed package

DEFAULT_METHOD = get_settings().preprocess.default_method
MIN_SAMPLES = get_settings().preprocess.min_samples
META_FILE = get_settings().inference.meta_file

METHOD = DEFAULT_METHOD
MODEL_DIR = ROOT / "models" / "anomaly_baseline"

## 1. Data and helpers

The real datasets, fault injection and the dataset builder (#9, #95).

Now in `src/guardian_layer/ai/datasets/` (#95): one loader per source, registered in
`DatasetFactory`, with its settings in `config/datasets.yaml`. The Intel Berkeley Lab
loader (formerly the `research/intel_lab.py` script, then this section) is
`intel_lab.py`. Sensor ids are `<source>:<station>:<type>`, e.g.
`intel_lab:1:temperature`.

Intel Berkeley Research Lab: 54 Mica2Dot motes, Feb 28 - Apr 5 2004, one reading every
~31 s. Source: https://db.csail.mit.edu/labdata/labdata.html — free to use; the
authors ask to be acknowledged in published work.

In [ ]:
DATASET_SETTINGS = get_dataset_settings()
INJECTION = DATASET_SETTINGS.injection
# type -> (unit, lower_range, upper_range) of the Intel Lab types (SETUPS, TYPE_CODES)
TYPES = DATASET_SETTINGS.intel_lab.types

Now in `src/guardian_layer/ai/datasets/injection.py` (`FaultInjector`, #95), formerly
the `research/injection.py` script, then this section; the kinds, labels and fault
sizes are in `config/datasets.yaml` (`injection`).

Inject labelled sensor faults (label 1) into clean real readings. Sizes are multiples
of each type's normal within-window spread, so a fault means the same thing for every
sensor type.

In [ ]:
FAULT_KINDS = INJECTION.fault_kinds
# #55: streams that stay in range but do not behave like a physical sensor
FABRICATED_KINDS = INJECTION.fabricated_kinds
TRAIN_KINDS = (*FAULT_KINDS, *FABRICATED_KINDS)
# the label each kind trains the model on; "spoof" has none: the static check owns
# spoofed_data, so spoof windows are kept out of the model and scored by the rule
KIND_LABELS = INJECTION.labels

Now in `src/guardian_layer/ai/datasets/windows.py` (`LabelledWindows`, #95), formerly
the `research/common.py` script, then this section: one feature row per sensor-window
(`window_slots` sample periods, non-overlapping) with the real PreProcess, a binary
label (1 = injected fault) and the train/test split. `read` turns a source into
columns once (slow, cached below); `build` injects faults with one seed and builds
the features. The metrics helpers stay here.

In [ ]:
WINDOW_SECONDS = 20  # the generator's windows (sections 5-7)
INJECTION_SEEDS = (0, 1, 2)
MODEL_SEEDS = (0, 1, 2, 3, 4)
# memory: each worker loads the cached readings of every source in DATASETS, so it
# grows with the sources (~0.7 GB with the Intel Lab alone)
MAX_WORKERS = 8

`read` takes minutes to hours per source, so `source_readings` caches its result in
`~/.cache/guardian_layer/readings/<source>.npz` (or under `$GUARDIAN_DATA_DIR`), and
reads again only when a setting `read` depends on changes (`sample_seconds`,
`max_readings`, `fast_sensor_share`, and for a sampled source `sample_every_hours` and
`block_hours`: an unsampled source ignores both, so section 9's long settings reuse its
file). **Delete the file after
changing a loader or a data file**: the cache cannot see either.

In [ ]:
FAST_SENSORS: dict[str, list[str]] = {}  # source -> sensors read dropped as too fast


def read_key(name: str, settings: DatasetSettings) -> dict[str, Any]:
    """The settings ``read`` depends on, as the cache file stores them."""
    injection = settings.injection
    every_hours = injection.sample_every_hours.get(name, 1)
    wanted: dict[str, Any] = {
        "sample_seconds": injection.sample_seconds.get(name),
        # not in SourceReadings: stored next to it in the cache file
        "max_readings": injection.max_readings,
        "fast_sensor_share": injection.fast_sensor_share,
    }
    if every_hours > 1:  # the blocks shape only a sampled read
        wanted |= {"every_hours": every_hours, "block_hours": injection.block_hours}
    return wanted


def source_readings(
    name: str, settings: DatasetSettings = DATASET_SETTINGS, suffix: str = ""
) -> SourceReadings:
    """The source's readings, from the cache when read with these settings.

    ``suffix`` names a second cache file, for a source read another way (section 9's
    Array of Things in 24 h blocks): one file never holds two reads in turn.
    """
    default = Path.home() / ".cache" / "guardian_layer"
    root = Path(os.environ.get("GUARDIAN_DATA_DIR", default))
    path = root / "readings" / f"{name}{suffix}.npz"
    wanted = read_key(name, settings)
    if path.exists():
        with np.load(path) as cached:
            if all(
                key in cached and cached[key].item() == value
                for key, value in wanted.items()
            ):
                FAST_SENSORS[name] = cached["fast_sensors"].tolist()
                return SourceReadings.from_arrays(cached)

    # read only logs the sensors it drops as too fast: catch them for the table
    caught = logging.handlers.BufferingHandler(capacity=1_000_000)
    windows_log = logging.getLogger("guardian_layer.ai.datasets.windows")
    windows_log.addHandler(caught)
    try:
        data = LabelledWindows(settings).read(name, DatasetFactory(settings).load(name))
    finally:
        windows_log.removeHandler(caught)
    FAST_SENSORS[name] = [
        sensor for record in caught.buffer for sensor in getattr(record, "sensors", [])
    ]
    path.parent.mkdir(parents=True, exist_ok=True)
    partial_path = path.with_name(
        path.name + ".part"
    )  # an interrupted save is not cached
    with partial_path.open("wb") as file:
        np.savez(
            file,
            fast_sensors=np.array(FAST_SENSORS[name], dtype=np.str_),
            max_readings=np.array(settings.injection.max_readings),
            fast_sensor_share=np.array(settings.injection.fast_sensor_share),
            **data.to_arrays(),
        )
    os.replace(partial_path, path)
    return data

In [ ]:
# self-check (instant): the long settings change the read key of the sampled source
# only, so the 18 unsampled sources reuse their files
_long = DATASET_SETTINGS.long_window(21600.0)
assert read_key("bellinge", _long) == read_key("bellinge", DATASET_SETTINGS)
assert read_key("array_of_things", _long) != read_key(
    "array_of_things", DATASET_SETTINGS
)
assert "block_hours" not in read_key("bellinge", DATASET_SETTINGS)

In [ ]:
# the sources build_dataset joins; Intel Lab alone keeps the results comparable with
# the committed results.md files
DATASETS = ("intel_lab",)


def build_dataset(
    method: str,
    injection_seed: int,
    kinds: tuple[str, ...] = FAULT_KINDS,
    process: Callable[..., dict[str, dict[str, float]]] = process_window,
    columns: tuple[str, ...] | None = None,
    datasets: tuple[str, ...] = DATASETS,
) -> WindowDataset:
    """Features of every sensor-window of these sources, after injecting faults with
    this seed: LabelledWindows.build per source, joined.

    ``process`` and ``columns`` swap in another feature builder (section 7).
    """
    builder = LabelledWindows(method=method)
    parts = [
        builder.build(source_readings(name), injection_seed, kinds, process, columns)
        for name in datasets
    ]
    return join_parts(parts, list(datasets))


def join_parts(parts: list[WindowDataset], names: list[str]) -> WindowDataset:
    """Several sources' (or runs') WindowDatasets as one, ``names`` one per part."""
    arrays = [
        f.name
        for f in dataclasses.fields(WindowDataset)
        if f.name not in ("spreads", "window")
    ]
    # window numbers restart at 0 per source: shift each source past the one before,
    # so code keyed on window numbers never mixes sources
    offsets = np.cumsum([0, *(part.window.max(initial=-1) + 1 for part in parts[:-1])])
    return WindowDataset(
        **{f: np.concatenate([getattr(part, f) for part in parts]) for f in arrays},
        window=np.concatenate(
            [part.window + offset for part, offset in zip(parts, offsets, strict=True)]
        ),
        # one spread per source and type, e.g. "intel_lab:temperature"
        spreads={
            f"{name}:{kind}": spread
            for name, part in zip(names, parts, strict=True)
            for kind, spread in part.spreads.items()
        },
    )

In [ ]:
def report(
    truth: BoolArray,
    kind: npt.NDArray[np.str_],
    scores: FloatArray,
    predicted: BoolArray,
) -> dict[str, float]:
    """Precision/recall/F1/PR-AUC, plus recall per injected fault kind."""
    result = {
        "precision": precision_score(truth, predicted, zero_division=0),
        "recall": recall_score(truth, predicted, zero_division=0),
        "f1": f1_score(truth, predicted, zero_division=0),
        "pr_auc": average_precision_score(truth, scores),
    }
    for fault_kind in FAULT_KINDS:
        hit = kind == fault_kind
        result[f"recall_{fault_kind}"] = (
            float(predicted[hit].mean()) if hit.any() else 0.0
        )
    return {name: float(value) for name, value in result.items()}

In [ ]:
def mean_sd(runs: list[dict[str, float]]) -> dict[str, str]:
    """Each metric as 'mean ± sd' over runs (sample sd; 0 for one run)."""
    cells = {}
    for name in runs[0]:
        values = np.array([run[name] for run in runs])
        sd = values.std(ddof=1) if len(values) > 1 else 0.0
        cells[name] = f"{values.mean():.3f} ± {sd:.3f}"
    return cells

In [ ]:
def markdown_table(rows: dict[str, dict[str, str]]) -> str:
    """Rows of named string cells as a markdown table."""
    columns = list(next(iter(rows.values())))
    lines = ["| | " + " | ".join(columns) + " |", "|---" * (len(columns) + 1) + "|"]
    for name, cells in rows.items():
        lines.append(f"| {name} | " + " | ".join(cells[c] for c in columns) + " |")
    return "\n".join(lines)

In [ ]:
def fit_isolation_forest(
    train: FloatArray, truth: BoolArray, test: FloatArray, seed: int
) -> tuple[FloatArray, BoolArray]:
    """Unsupervised: contamination = training positive share (IF allows <= 0.5)."""
    model = IsolationForest(
        random_state=seed, contamination=min(0.5, float(truth.mean()))
    ).fit(train)
    return -model.score_samples(test), model.predict(test) == -1

In [ ]:
def fit_xgboost(
    train: FloatArray, truth: BoolArray, test: FloatArray, seed: int
) -> tuple[FloatArray, BoolArray]:
    """Supervised binary (0 correct / 1 wrong), positives up-weighted."""
    positives = max(int(truth.sum()), 1)
    model = XGBClassifier(
        random_state=seed,
        n_jobs=1,  # one process per task already runs in parallel
        scale_pos_weight=(len(truth) - positives) / positives,
    ).fit(train, truth)
    scores = model.predict_proba(test)[:, 1]
    return scores, scores > 0.5

## Real datasets

Every registered source through `LabelledWindows` (#95), injected with seed 0: Intel
Lab first, Array of Things second, then the rest. NAB is reported only: it never
trains. A source whose files are not under `datasets.root` is listed as `missing`,
and one with no measured sample period (`injection.sample_seconds`) as
`not configured`. Sampled hours are `block_hours/sample_every_hours` for the sources
read in hour blocks, else `all`.

In [ ]:
REPORTED_ONLY = ("nab",)  # listed in the table, never trained on
FIRST_SOURCES = ("intel_lab", "array_of_things")


def datasets_main(method: str = METHOD, seed: int = 0) -> None:
    names = [
        *FIRST_SOURCES,
        *(name for name in DatasetFactory.names() if name not in FIRST_SOURCES),
    ]
    by_label: dict[str, list[str]] = {}  # label -> its kinds; spoof has no label
    for kind in TRAIN_KINDS:
        label = str(KIND_LABELS[kind]) if kind in KIND_LABELS else "no label"
        by_label.setdefault(label, []).append(kind)
    rows: dict[str, dict[str, str]] = {}
    for name in names:
        title = f"{name} (reported only)" if name in REPORTED_ONLY else name
        try:
            rows[title] = dataset_row(name, method, seed, by_label)
        except FileNotFoundError:
            rows[title] = {"status": "missing"}
        except ValueError as error:  # e.g. no reading left, no eligible sensor-window
            # "|" would split the table cell
            rows[title] = {"status": f"error: {error}".replace("|", "/")}
        # progress: the first read of a big source takes hours
        print(f"{title}: " + ", ".join(f"{k} {v}" for k, v in rows[title].items()))
    # markdown_table takes its columns from the first row; fill the short rows
    columns = max(rows.values(), key=len)
    print(
        markdown_table(
            {name: dict.fromkeys(columns, "") | row for name, row in rows.items()}
        )
    )


def dataset_row(
    name: str, method: str, seed: int, by_label: dict[str, list[str]]
) -> dict[str, str]:
    """One source's row of the datasets table."""
    if name not in INJECTION.sample_seconds:
        return {"status": "not configured"}
    started = perf_counter()
    data = source_readings(name)
    read_seconds = perf_counter() - started
    started = perf_counter()
    built = LabelledWindows(method=method).build(data, seed, TRAIN_KINDS)
    build_seconds = perf_counter() - started
    every = int(data.info.every_hours)
    return {
        "status": "ok",
        "sample period": f"{float(data.info.sample_seconds):g} s",
        "sampled hours": (
            f"{int(data.info.block_hours)}/{every}" if every > 1 else "all"
        ),
        "readings kept": f"{len(data.readings.slot):,}",
        "dropped fast sensors": str(len(FAST_SENSORS[name])),
        "windows": f"{len(np.unique(built.window)):,}",
        "sensor-windows": f"{len(built.kind):,}",
        **{
            f"injected {label}": f"{int(np.isin(built.kind, kinds).sum()):,}"
            for label, kinds in by_label.items()
        },
        "real faults": f"{int(built.real_fault.sum()):,}",
        "events": f"{int(built.event.sum()):,}",
        "read": f"{read_seconds:.0f} s",
        "build": f"{build_seconds:.0f} s",
    }

Long-running: the first read of every source takes hours (Array of Things alone
about 1 h); later runs read the cache. Uses `METHOD`.

In [ ]:
datasets_main()

## 2. Decoupling comparison

Does each decoupling method generalize across sensor types? (#9)

Moved from the former `research/decoupling/compare.py` script.

Experiment 1: does each decoupling method generalize across sensor types?

Intel Berkeley Lab data with injected faults (label 1). For each method and model:
train on two sensor types and test on the third (repeated for every type), plus an
all-types reference. The generalization gap is all-types score minus mean unseen-
type score. Runs (method, injection seed) tasks in parallel processes.

In [ ]:
METHODS = ("none", "fft", "haar", "db4", "ssa")
SETUPS = ("all types", *(f"unseen {name}" for name in TYPES))
COMPARE_MODELS = {
    "isolation_forest": (fit_isolation_forest, MODEL_SEEDS),
    "xgboost": (fit_xgboost, (0,)),  # deterministic for a given dataset
}

CompareRuns = dict[tuple[str, str], list[dict[str, float]]]

In [ ]:
def compare_run(task: tuple[str, int]) -> tuple[str, CompareRuns]:
    """All setups and models for one (method, injection seed)."""
    method, injection_seed = task
    data = build_dataset(method, injection_seed)
    usable = ~data.real_fault & ~data.event  # kept apart: real faults, real events
    runs: CompareRuns = {}
    for setup in SETUPS:
        train, test = data.train & usable, ~data.train & usable
        if setup != "all types":
            held_out = setup.split()[1]
            train &= data.sensor_type != held_out
            test &= data.sensor_type == held_out
        for model, (fit, seeds) in COMPARE_MODELS.items():
            for seed in seeds:
                scores, predicted = fit(
                    data.features[train], data.label[train], data.features[test], seed
                )
                runs.setdefault((model, setup), []).append(
                    report(data.label[test], data.kind[test], scores, predicted)
                )
    return method, runs

In [ ]:
def summary_row(all_types: list[dict[str, float]], unseen: list[dict[str, float]]):
    """All-types vs unseen-type F1 / PR-AUC (mean ± sd) and the gap of the means."""
    row = {}
    for metric in ("f1", "pr_auc"):
        gap = np.mean([r[metric] for r in all_types]) - np.mean(
            [r[metric] for r in unseen]
        )
        row[f"all types {metric}"] = mean_sd(all_types)[metric]
        row[f"unseen type {metric}"] = mean_sd(unseen)[metric]
        row[f"gap {metric}"] = f"{gap:.3f}"
    return row

In [ ]:
def compare_main() -> None:
    tasks = [(method, seed) for method in METHODS for seed in INJECTION_SEEDS]
    results: dict[str, CompareRuns] = {method: {} for method in METHODS}
    for name in DATASETS:  # read once, so the workers do not each read the sources
        source_readings(name)
    for method, runs in Parallel(n_jobs=min(len(tasks), MAX_WORKERS))(
        delayed(compare_run)(task) for task in tasks
    ):
        for key, reports in runs.items():
            results[method].setdefault(key, []).extend(reports)

    for model in COMPARE_MODELS:
        summary = {
            method: summary_row(
                results[method][(model, "all types")],
                [r for s in SETUPS[1:] for r in results[method][(model, s)]],
            )
            for method in METHODS
        }
        print(f"\n## {model}: summary\n")
        print(markdown_table(summary))
        detail = {
            f"{method} / {setup}": mean_sd(results[method][(model, setup)])
            for method in METHODS
            for setup in SETUPS
        }
        print(f"\n## {model}: detail\n")
        print(markdown_table(detail))

Long-running: 15 tasks (5 methods x 3 injection seeds) in up to 8 worker processes.

In [ ]:
compare_main()

## 3. General-model exploration

One model for all sensor types, and the real-fault check. (#9)

Moved from the former `research/general_model/explore.py` script.

Explore training ONE anomaly model for all sensor types, and check real faults.

Intel Berkeley Lab data with injected faults (label 1), one decoupling method.
Models: Isolation Forest and LOF (no labels needed), XGBoost and logistic regression
(trained on the 0/1 labels). All-types train/test, plus the real-fault check: share
of real battery-fault windows flagged as 1, and false alarms on normal test windows.

In [ ]:
LOF_MAX_TRAIN = 20_000  # LOF is a k-NN method; subsample to keep it fast

Fit = Callable[[FloatArray, BoolArray, FloatArray, int], tuple[FloatArray, BoolArray]]

In [ ]:
def fit_lof(
    train: FloatArray, truth: BoolArray, test: FloatArray, seed: int
) -> tuple[FloatArray, BoolArray]:
    """Unsupervised density model on a random training subsample."""
    rng = np.random.default_rng(seed)
    pick = rng.choice(len(train), size=min(LOF_MAX_TRAIN, len(train)), replace=False)
    scaler = StandardScaler().fit(train[pick])
    model = LocalOutlierFactor(
        novelty=True, contamination=min(0.5, float(truth[pick].mean()))
    ).fit(scaler.transform(train[pick]))
    scaled = scaler.transform(test)
    return -model.score_samples(scaled), model.predict(scaled) == -1

In [ ]:
def fit_logistic(
    train: FloatArray, truth: BoolArray, test: FloatArray, seed: int
) -> tuple[FloatArray, BoolArray]:
    """Supervised linear baseline on standardized features."""
    scaler = StandardScaler().fit(train)
    model = LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=seed
    ).fit(scaler.transform(train), truth)
    scores = model.predict_proba(scaler.transform(test))[:, 1]
    return scores, scores > 0.5

In [ ]:
EXPLORE_MODELS: dict[str, tuple[Fit, tuple[int, ...]]] = {
    "isolation_forest": (fit_isolation_forest, MODEL_SEEDS),
    "lof": (fit_lof, MODEL_SEEDS),
    "xgboost": (fit_xgboost, (0,)),
    "logistic_regression": (fit_logistic, (0,)),
}

ExploreRuns = dict[str, list[dict[str, float]]]

In [ ]:
def explore_run(method: str, injection_seed: int) -> tuple[ExploreRuns, ExploreRuns]:
    """Test metrics and real-fault check per model, for one injection seed."""
    data = build_dataset(method, injection_seed)
    usable = ~data.real_fault & ~data.event
    train, test = data.train & usable, ~data.train & usable
    normal_test = test & ~data.label
    scored = test | data.real_fault  # one prediction pass covers both checks
    metrics: ExploreRuns = {}
    checks: ExploreRuns = {}
    for model, (fit, seeds) in EXPLORE_MODELS.items():
        for seed in seeds:
            scores, predicted = fit(
                data.features[train], data.label[train], data.features[scored], seed
            )
            in_test = test[scored]
            metrics.setdefault(model, []).append(
                report(
                    data.label[test],
                    data.kind[test],
                    scores[in_test],
                    predicted[in_test],
                )
            )
            checks.setdefault(model, []).append(
                {
                    "real_faults_flagged": float(
                        predicted[data.real_fault[scored]].mean()
                    ),
                    "false_alarms": float(predicted[normal_test[scored]].mean()),
                }
            )
    return metrics, checks

In [ ]:
def explore_main(method: str) -> None:
    metrics: ExploreRuns = {}
    checks: ExploreRuns = {}
    for name in DATASETS:  # read once, so the workers do not each read the sources
        source_readings(name)
    workers = min(len(INJECTION_SEEDS), MAX_WORKERS)
    for run_metrics, run_checks in Parallel(n_jobs=workers)(
        delayed(explore_run)(method, seed) for seed in INJECTION_SEEDS
    ):
        for model, reports in run_metrics.items():
            metrics.setdefault(model, []).extend(reports)
        for model, reports in run_checks.items():
            checks.setdefault(model, []).extend(reports)
    print(f"decoupling method: {method}\n")
    print("## Injected faults, all types\n")
    print(markdown_table({model: mean_sd(runs) for model, runs in metrics.items()}))
    print("\n## Real battery faults\n")
    print(markdown_table({model: mean_sd(runs) for model, runs in checks.items()}))

Long-running: 3 injection seeds in parallel worker processes. Uses `METHOD`.

In [ ]:
explore_main(METHOD)

## 4. Baseline training

Trains the #10 baseline model and writes it to `MODEL_DIR` (git-ignored).

Moved from the former `research/baseline/train.py` script.

Train the #10 baseline anomaly model and write the artifact.

Intel Berkeley Lab data with injected faults (label 1), same split as the general-
model exploration section. Writes model.json, meta.json and metrics.json.

In [ ]:
INJECTION_SEED = 0

In [ ]:
def train_main(method: str = DEFAULT_METHOD, out: Path = MODEL_DIR) -> None:
    data = build_dataset(method, INJECTION_SEED, TRAIN_KINDS)
    classes = get_settings().inference.classes
    names = ["normal", *(str(label) for label in classes)]
    # class index per row: 0 normal, i = classes[i - 1]
    class_index = np.array(
        [
            classes.index(KIND_LABELS[kind]) + 1 if kind in KIND_LABELS else 0
            for kind in data.kind
        ]
    )
    spoof = data.kind == "spoof"  # no model class: scored by the static check below
    usable = ~data.real_fault & ~data.event & ~spoof
    train, test = data.train & usable, ~data.train & usable
    model = BaselineModel.fit(
        data.features[train],
        class_index[train],
        method,
        get_settings().inference.default_threshold,
        PreProcess().feature_keys(method),
        classes,
    )

    scores, anomaly_class = model.predict(data.features[test])
    flagged = scores > model.threshold
    predicted = np.where(flagged, anomaly_class, 0)
    truth = class_index[test]
    metrics = report(truth > 0, data.kind[test], scores, flagged)
    for kind in FABRICATED_KINDS:
        hit = data.kind[test] == kind
        metrics[f"recall_{kind}"] = float(flagged[hit].mean()) if hit.any() else 0.0
    for index, name in enumerate(names):
        is_class, said_class = truth == index, predicted == index
        metrics[f"precision_{name}"] = float(
            precision_score(is_class, said_class, zero_division=0)
        )
        metrics[f"recall_{name}"] = float(
            recall_score(is_class, said_class, zero_division=0)
        )
        metrics[f"f1_{name}"] = float(f1_score(is_class, said_class, zero_division=0))

    # the spoofed_data rule on held-out windows the model never saw
    check = StaticCheck()
    # held-out rows are not real alarms: keep them out of the logs
    logging.getLogger("guardian_layer.pipelines.static_check").setLevel(logging.ERROR)

    def rule_share(rows: BoolArray) -> float:
        hits = [
            check.spoofed_data("held-out", dict(zip(model.keys, row, strict=True)))
            is Anomalies.SPOOFED_DATA
            for row in data.features[rows]
        ]
        return float(np.mean(hits)) if hits else 0.0

    held_out = ~data.train & ~data.real_fault & ~data.event
    metrics["rule_recall_spoof"] = rule_share(held_out & spoof)
    metrics["rule_flag_rate_normal"] = rule_share(held_out & (data.kind == ""))

    model.save(out)
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print(json.dumps(metrics, indent=2))
    print(f"\nconfusion matrix, rows = truth, columns = predicted, order {names}:")
    print(confusion_matrix(truth, predicted, labels=list(range(len(names)))))

Long-running: builds the dataset and trains. Uses `METHOD`.

In [ ]:
train_main(METHOD)

## 5. Evaluation

Scores the model on the project's own sensor types from the flood-scenario
generator. (#12)

In [ ]:
from mocks.telemetry_generator import (
    LABELS,
    PACKET_ANOMALIES,
    SENSOR_TYPES,
    VALUE_ANOMALIES,
    generate,
)

Moved from the former `research/evaluation/evaluate.py` script.

Evaluate the baseline anomaly model on the flood-scenario generator (#12).

The model learned on Intel Lab data (temperature, humidity, light); this scores it
on the project's own sensor types from mocks/telemetry_generator.py. Each seed's
output is cut into non-overlapping 20 s windows in arrival order, as the Guardian
buffer receives it. A sensor-window is label 1 when it holds a value fault
(out_of_range, spoofed, stuck); packet problems (duplicate, replayed, out_of_order,
packet_gap) stay 0, as in the research, and the share of them flagged is reported
apart.

The model is compared with two rules on the same windows (a value outside its range,
or the same value STUCK_RUN times in a row) and with rules + model, the designed
pipeline where the model is the last resort. Cells are mean ± sd over the seeds.

model_dir defaults to MODEL_DIR, written by the baseline training section.

In [ ]:
SEEDS = (0, 1, 2)
SENSORS = 20  # generator default
DURATION = 3600  # seconds per seed: 180 windows
ANOMALY_RATE = 0.01  # ~10% of sensor-windows get a value fault, like the research's 9%

In [ ]:
STUCK_RUN = 3  # the same value this many times in a row counts as stuck
THRESHOLDS = tuple(t / 10 for t in range(1, 10))

In [ ]:
Columns = dict[str, npt.NDArray[Any]]
Metrics = dict[str, float]

In [ ]:
def sensor_windows(
    telemetry: list[Sensor], labels: list[dict[str, Any]]
) -> list[tuple[list[Sensor], dict[str, set[str]]]]:
    """Non-overlapping 20 s windows in arrival order, with the line labels per sensor.

    A line's arrival time is the newest timestamp seen so far: lines come in arrival
    order, and a replayed or delayed message carries an older timestamp than that.
    """
    preprocessor = PreProcess()
    windows: list[tuple[list[Sensor], dict[str, set[str]]]] = []
    start = newest = preprocessor.timestamp_seconds(telemetry[0])
    for reading, row in zip(telemetry, labels, strict=True):
        newest = max(newest, preprocessor.timestamp_seconds(reading))
        index = int((newest - start) // WINDOW_SECONDS)
        while len(windows) <= index:
            windows.append(([], {}))
        window, kinds = windows[index]
        window.append(reading)
        kinds.setdefault(reading.sensor_id, set()).add(row["label"])
    return windows

In [ ]:
def longest_run(values: npt.NDArray[np.float64]) -> int:
    """Length of the longest stretch of equal consecutive values."""
    best = run = 1
    for before, after in pairwise(values):
        run = run + 1 if after == before else 1
        best = max(best, run)
    return best

In [ ]:
def rule_flags(window: list[Sensor]) -> dict[str, bool]:
    """Per sensor: a value outside its range, or STUCK_RUN equal values in a row.

    Same dedup and range lookup as PreProcess, so both see the same readings.
    """
    preprocessor = PreProcess()
    unique, _ = preprocessor.deduplicate(window)
    by_sensor: dict[str, list[Sensor]] = {}
    by_cluster: dict[str, list[Sensor]] = {}
    for reading in unique:
        by_sensor.setdefault(reading.sensor_id, []).append(reading)
        by_cluster.setdefault(preprocessor.cluster_key(reading), []).append(reading)
    flags = {}
    for sensor_id, readings in by_sensor.items():
        _, values = preprocessor.sensor_series(readings)
        value_range = preprocessor.find_range(
            readings, by_cluster[preprocessor.cluster_key(readings[0])]
        )
        outside = value_range is not None and bool(
            ((values < value_range[0]) | (values > value_range[1])).any()
        )
        flags[sensor_id] = outside or longest_run(values) >= STUCK_RUN
    return flags

In [ ]:
def score_windows(model: BaselineModel, seed: int) -> Columns:
    """One row per sensor-window: type, model score, the model's flag (its
    threshold for the sensor's rate, as live), rule flag, one column per label."""
    telemetry, labels = generate(
        seed=seed, sensors=SENSORS, duration=DURATION, anomaly_rate=ANOMALY_RATE
    )
    readings = [Sensor.model_validate(message) for message in telemetry]
    method = model.method
    preprocessor, decompose = PreProcess(method), Decompose()
    types, scores, flagged, rules = [], [], [], []
    has: dict[str, list[bool]] = {label: [] for label in LABELS}
    for window, kinds in sensor_windows(readings, labels):
        flags = rule_flags(window)
        features = process_window(preprocessor, decompose, method, window)
        for sensor_id, verdict in model_verdicts(
            model, features, list(features)
        ).items():
            types.append(sensor_id.rsplit("-", 1)[0])
            scores.append(verdict.score)
            flagged.append(verdict.anomalous)
            rules.append(flags[sensor_id])
            for label in LABELS:
                has[label].append(label in kinds[sensor_id])
    return {
        "type": np.array(types),
        "score": np.array(scores, dtype=np.float64),
        "flag": np.array(flagged),
        "rule": np.array(rules),
        **{label: np.array(rows) for label, rows in has.items()},
    }

In [ ]:
def share(flagged: BoolArray, rows: BoolArray) -> float:
    """Share of the given rows that are flagged (nan when there are none)."""
    return float(flagged[rows].mean()) if rows.any() else float("nan")

In [ ]:
def base(truth: BoolArray, flagged: BoolArray) -> Metrics:
    return {
        "precision": float(precision_score(truth, flagged, zero_division=0)),
        "recall": float(recall_score(truth, flagged, zero_division=0)),
        "f1": float(f1_score(truth, flagged, zero_division=0)),
        "false_alarms": share(flagged, ~truth),
    }

In [ ]:
def evaluate_seed(model: BaselineModel, seed: int) -> dict[str, dict[str, Metrics]]:
    """Every table's rows for one seed."""
    data = score_windows(model, seed)
    truth = np.any([data[kind] for kind in VALUE_ANOMALIES], axis=0)
    packet = np.any([data[kind] for kind in PACKET_ANOMALIES], axis=0)
    # the model's own verdicts: a model with a threshold per rate judges each
    # sensor at its rate's (for one without, the same as score > threshold)
    model = data["flag"]
    systems = {
        "model": model,
        "rules": data["rule"],
        "rules + model": data["rule"] | model,
    }
    types = {"all": np.ones_like(truth)} | {t: data["type"] == t for t in SENSOR_TYPES}
    no_value_fault = {"clean": ~truth & ~packet} | {
        kind: data[kind] & ~truth for kind in PACKET_ANOMALIES
    }
    return {
        "Model vs rules, all sensor types": {
            name: base(truth, flagged)
            | {f"recall_{kind}": share(flagged, data[kind]) for kind in VALUE_ANOMALIES}
            for name, flagged in systems.items()
        },
        "Model by sensor type": {
            name: {"positive_share": float(truth[rows].mean())}
            | base(truth[rows], model[rows])
            | {
                "pr_auc": float(
                    average_precision_score(truth[rows], data["score"][rows])
                )
            }
            for name, rows in types.items()
        },
        "Share flagged among windows without a value fault": {
            name: {"model": share(model, rows), "rules": share(data["rule"], rows)}
            for name, rows in no_value_fault.items()
        },
        "Model threshold sweep, all sensor types": {
            f"{threshold:.1f}": base(truth, data["score"] > threshold)
            for threshold in THRESHOLDS
        },
    }

In [ ]:
def evaluate_main(model_dir: Path = MODEL_DIR) -> None:
    if not (model_dir / META_FILE).exists():
        sys.exit(f"no model in {model_dir}: run the baseline training section first")
    model = BaselineModel.load(model_dir)
    tables: dict[str, dict[str, list[Metrics]]] = {}
    for seed in SEEDS:
        for title, rows in evaluate_seed(model, seed).items():
            for name, metrics in rows.items():
                tables.setdefault(title, {}).setdefault(name, []).append(metrics)
    print(
        f"model: {model_dir} (method {model.method}, threshold "
        f"{model.threshold}); generator: seeds {SEEDS}, {SENSORS} sensors, "
        f"{DURATION} s each, anomaly rate {ANOMALY_RATE}"
    )
    for title, table in tables.items():
        print(f"\n## {title}\n")
        print(markdown_table({name: mean_sd(runs) for name, runs in table.items()}))

Needs the model from section 4.

In [ ]:
evaluate_main()

## 6. Latency benchmark

Inference latency per 20 s sensor window. (#12)

Moved from the former `research/evaluation/benchmark.py` script.

Inference latency per 20 s sensor window (#12), split into preprocessing, static checks and model.

Each window is one 20 s generator run (mocks/telemetry_generator.py): N sensors, one
reading per second each, with the generator's packet problems and value faults.
Times are wall-clock (time.perf_counter) in one process, after WARMUP untimed runs;
the model load and the first (cold) call are timed apart. A new window arrives every
SLIDE_SECONDS, so each one must be processed well within that.

Without a trained model in model_dir (default MODEL_DIR), a stand-in is fitted on
random features with the same XGBoost settings: same tree count and depth limit, so
scoring costs about the same. Report the numbers of the real model.

In [ ]:
SENSOR_COUNTS = (10, 100, 500, 1000, 2000, 5000)
WARMUP = 3
REPEATS = 50  # timed runs per sensor count; p99 of 50 runs is about the slowest one

In [ ]:
SLIDE_SECONDS = 10  # SensorBuffer.SLIDE_SECONDS: a new window every 10 s

In [ ]:
def stand_in() -> BaselineModel:
    """The baseline's XGBoost settings, fitted on random rows (for timing only)."""
    rng = np.random.default_rng(0)
    keys = PreProcess().feature_keys(DEFAULT_METHOD)
    inference = get_settings().inference
    features = rng.normal(size=(5_000, len(keys)))
    labels = rng.integers(0, len(inference.classes) + 1, 5_000)  # plus normal
    return BaselineModel.fit(
        features,
        labels,
        DEFAULT_METHOD,
        inference.default_threshold,
        keys,
        inference.classes,
    )

In [ ]:
def ms(seconds: list[float], percentile: float) -> float:
    return float(np.percentile(seconds, percentile)) * 1000

In [ ]:
def static_checks(
    preprocessor: PreProcess,
    checker: StaticCheck,
    window: list[Sensor],
    features: dict[str, dict[str, float]],
) -> None:
    """Every per-window rule check on the raw window (timed, not kept).

    Dedup is timed in preprocess, so it is not repeated here.
    """
    by_sensor: dict[str, list[Sensor]] = {}
    by_cluster: dict[str, list[Sensor]] = {}
    for reading in window:
        by_sensor.setdefault(reading.sensor_id, []).append(reading)
        by_cluster.setdefault(reading.cluster_id, []).append(reading)
    window_end = max(reading.timestamp for reading in window)
    for sensor_id, readings in by_sensor.items():
        cluster = by_cluster[readings[0].cluster_id]
        checker.out_of_range(readings, preprocessor.find_range(readings, cluster))
        checker.sensor_is_down(readings, window_end)
        # a sensor whose readings were all dropped has no row: NOT_CHECKED
        checker.spoofed_data(sensor_id, features.get(sensor_id, {}))

In [ ]:
def benchmark_main(model_dir: Path = MODEL_DIR) -> None:
    real = (model_dir / META_FILE).exists()
    started = perf_counter()
    model = BaselineModel.load(model_dir) if real else stand_in()
    load_ms = (perf_counter() - started) * 1000
    method = model.method
    preprocessor, decompose, checker = PreProcess(method), Decompose(), StaticCheck()
    # generator faults are not real alarms: keep them out of the logs
    logging.getLogger("guardian_layer.pipelines.static_check").setLevel(logging.ERROR)

    # validated up front: a Buffer already holds Sensor readings, so it is not timed
    windows = {
        count: [
            Sensor.model_validate(message)
            for message in generate(seed=0, sensors=count, duration=WINDOW_SECONDS)[0]
        ]
        for count in SENSOR_COUNTS
    }
    started = perf_counter()
    features = process_window(
        preprocessor, decompose, method, windows[SENSOR_COUNTS[0]]
    )
    model_verdicts(model, features, list(features))
    cold_ms = (perf_counter() - started) * 1000

    rows = {}
    for count, window in windows.items():
        preprocess, checks, scoring = [], [], []
        for run in range(WARMUP + REPEATS):
            started = perf_counter()
            features = process_window(preprocessor, decompose, method, window)
            middle = perf_counter()
            static_checks(preprocessor, checker, window, features)
            checked = perf_counter()
            model_verdicts(model, features, list(features))
            ended = perf_counter()
            if run >= WARMUP:
                preprocess.append(middle - started)
                checks.append(checked - middle)
                scoring.append(ended - checked)
        total = [p + c + s for p, c, s in zip(preprocess, checks, scoring, strict=True)]
        rows[f"{count} sensors"] = {
            "messages": str(len(window)),
            "preprocess p50": f"{ms(preprocess, 50):.1f} ms",
            "checks p50": f"{ms(checks, 50):.1f} ms",
            "model p50": f"{ms(scoring, 50):.1f} ms",
            "total p50": f"{ms(total, 50):.1f} ms",
            "total p95": f"{ms(total, 95):.1f} ms",
            "total p99": f"{ms(total, 99):.1f} ms",
            "sensors/s at p50": f"{count / ms(total, 50) * 1000:,.0f}",
            f"{SLIDE_SECONDS} s budget used at p99": (
                f"{ms(total, 99) / (SLIDE_SECONDS * 1000):.2%}"
            ),
        }

    model = str(model_dir) if real else "stand-in (random features, no model found)"
    print(
        f"model: {model}; machine: {platform.machine()}, {os.cpu_count()} CPUs, "
        f"Python {platform.python_version()}"
    )
    if real:
        print(f"model load: {load_ms:.1f} ms")
    print(f"first call, {SENSOR_COUNTS[0]} sensors (cold): {cold_ms:.1f} ms")
    print(f"timed runs per row: {REPEATS}, after {WARMUP} warm-up runs\n")
    print(markdown_table(rows))

Uses the model from section 4 if present, else a stand-in.

In [ ]:
benchmark_main()

## 7. Model optimization

Which changes make the baseline model better? (#64)

Each idea from #64 is compared with the section 4 baseline (multiclass XGBoost on
`feature_keys("none")`, values scaled by their type's range, one model for all types),
one change at a time, on the same injected data: `TRAIN_KINDS`, spoof windows kept out
of the model, 3 injection seeds. Most ideas are new columns or a different unit for the
values, so one table per seed holds everything they are computed from (`window_table`).

- **Normalization**: the unit the values are divided by before the features: the
  type's range (now), the type's normal within-window spread (`type noise`), the
  sensor's own spread (`sensor noise`), or the window's own std (`window z-score`, what
  PreProcess does for a sensor without a range).
- **One model per sensor type** instead of one for all.
- **A `type` column** (what the sensor measures), with each decoupling method.
- **Sensor fusion**: the sensor against its same-type neighbours (`nbr`: spread ratio,
  slope difference) and against the other sensors at its location (`col`: spread and
  slope ratios).
- **Others**: a threshold picked on validation data; spoofing as a model class, with the
  change of the spatial delta since the previous window (`delta_change`).

The variants trained here then score the flood-scenario generator (as in section 5),
where the baseline flags almost every window.

In [ ]:
DECOUPLING_KEYS = tuple(
    key
    for method in METHODS[1:]
    for key in get_settings().preprocess.method_keys[method]
)
EXTRA_KEYS = (  # per sensor-window, what the variants below are computed from
    "last",  # newest normalized value
    "slope",  # least-squares slope of the normalized values, per second
    "z_last",  # newest value z-scored inside its window
    "nbr_std",  # mean within-window std of the same-type neighbours
    "nbr_slope",  # their mean slope
    "nbr_z_last",  # their mean z_last
    "lat",
    "lon",
)
TABLE_KEYS = (*PreProcess().feature_keys("none"), *DECOUPLING_KEYS, *EXTRA_KEYS)
# how each feature changes when the values are divided by u instead of the range: x u^p
UNIT_POWER = {
    "mean": 1,
    "variance": 2,
    "rolling_variance_max": 2,
    "rate_of_change_mean_abs": 1,
    "rate_of_change_max_abs": 1,
    "spatial_delta_mean": 1,
    "spatial_delta_last": 1,
    **{key: 2 for key in DECOUPLING_KEYS if key.endswith("_energy")},
    **{key: 1 for key in DECOUPLING_KEYS if key.endswith("_max_abs")},
}
# spread units added to both sides of a ratio, so a flat window stays finite
RATIO_FLOOR = 0.01
TYPE_CODES = {name: float(code) for code, name in enumerate(TYPES)}

In [ ]:
def slope(times: FloatArray, values: FloatArray) -> float:
    """Least-squares slope per second (0 without two distinct times)."""
    centered = times - times.mean()
    spread = float(centered @ centered)
    return float(centered @ (values - values.mean())) / spread if spread else 0.0


def window_table(
    preprocessor: PreProcess, decompose: Decompose, method: str, window: list[Sensor]
) -> dict[str, dict[str, float]]:
    """process_window (method "none") plus every decoupling method and EXTRA_KEYS."""
    unique, counters = preprocessor.deduplicate(preprocessor.finite(window))
    by_sensor: dict[str, list[Sensor]] = {}
    by_cluster: dict[str, list[Sensor]] = {}
    for reading in unique:
        by_sensor.setdefault(reading.sensor_id, []).append(reading)
        by_cluster.setdefault(preprocessor.cluster_key(reading), []).append(reading)

    rows: dict[str, dict[str, float]] = {}
    summaries = []
    for sensor_id, readings in by_sensor.items():
        cluster = preprocessor.cluster_key(readings[0])
        value_range = preprocessor.find_range(readings, by_cluster[cluster])
        times, raw = preprocessor.sensor_series(readings)
        values = preprocessor.normalize(raw, value_range)
        grid = preprocessor.resample(times, values)
        lat, lon = preprocessor.location(readings)
        std = float(values.std())
        rows[sensor_id] = {
            **preprocessor.value_features(times, values, value_range is not None),
            **counters[sensor_id],
            **{k: v for m in METHODS[1:] for k, v in decompose.caller(m, grid).items()},
            "last": float(values[-1]),
            "slope": slope(times, values),
            # what normalize gives without a range: zeros for a flat window
            "z_last": (values[-1] - values.mean()) / std if np.ptp(values) else 0.0,
            "lat": math.nan if lat is None else lat,
            "lon": math.nan if lon is None else lon,
        }
        summaries.append(
            SensorSummary(
                sensor_id, cluster, lat, lon, float(values.mean()), float(values[-1])
            )
        )
    clusters: dict[str, list[SensorSummary]] = {}
    for summary in summaries:  # input order kept inside each cluster: tie order
        clusters.setdefault(summary.cluster, []).append(summary)
    for members in clusters.values():  # spatial_features, plus the peers' spread
        neighbours = preprocessor.cluster_neighbors(members)
        for summary, indices in zip(members, neighbours, strict=True):
            peers = [members[i] for i in indices]
            row = rows[summary.sensor_id]
            if not peers:
                row |= dict.fromkeys(preprocessor.settings.spatial_keys, 0.0)
                row |= dict.fromkeys(("nbr_std", "nbr_slope", "nbr_z_last"), math.nan)
                continue
            others = [rows[peer.sensor_id] for peer in peers]
            row |= {
                "spatial_delta_mean": summary.mean
                - sum(p.mean for p in peers) / len(peers),
                "spatial_delta_last": summary.last
                - sum(p.last for p in peers) / len(peers),
                "neighbor_count": float(len(peers)),
                "nbr_std": float(np.mean([math.sqrt(o["variance"]) for o in others])),
                "nbr_slope": float(np.mean([o["slope"] for o in others])),
                "nbr_z_last": float(np.mean([o["z_last"] for o in others])),
            }
    return rows

In [ ]:
# self-check on generator data (ranges, no range, no location, no metadata): the
# production columns are exactly what process_window gives
_window = [
    Sensor.model_validate(message)
    for message in generate(seed=0, sensors=SENSORS, duration=WINDOW_SECONDS)[0]
]
_base = process_window(PreProcess(), Decompose(), "none", _window)
_table = window_table(PreProcess(), Decompose(), "none", _window)
_keys = PreProcess().feature_keys("none")  # rows also carry the short-window keys
assert all(_table[s][k] == _base[s][k] for s in _base for k in _keys)
assert all(set(_table[s]) == set(TABLE_KEYS) for s in _table)

In [ ]:
def column(data: WindowDataset, key: str) -> FloatArray:
    return data.features[:, TABLE_KEYS.index(key)]


def spreads(data: WindowDataset, rows: BoolArray) -> tuple[FloatArray, FloatArray]:
    """Per row, the normal within-window std of its type and of its own sensor.

    Medians over ``rows`` with enough readings, a range and a non-flat window, in the
    unit the table is in (range-scaled); a sensor without such windows takes its type's.
    """
    std = np.sqrt(column(data, "variance"))
    usable = (
        rows
        & (column(data, "sample_count") >= MIN_SAMPLES)
        & (column(data, "has_range") == 1)
        & (std > 0)
    )
    type_spread = np.full(len(std), np.nan)
    for name in np.unique(data.sensor_type):
        own = usable & (data.sensor_type == name)
        type_spread[data.sensor_type == name] = np.median(std[own]) if own.any() else 1
    sensor_spread = type_spread.copy()
    for sensor_id in np.unique(data.sensor_id[usable]):
        own = data.sensor_id == sensor_id
        sensor_spread[own] = np.median(std[usable & own])
    return type_spread, sensor_spread


def ratio(numerator: FloatArray, denominator: FloatArray) -> FloatArray:
    return np.log((numerator + RATIO_FLOOR) / (denominator + RATIO_FLOOR))


def co_located(data: WindowDataset, values: FloatArray) -> FloatArray:
    """Mean of ``values`` over the other sensors at this place in this window."""
    place = np.stack([data.window, column(data, "lat"), column(data, "lon")], axis=1)
    _, group = np.unique(place, axis=0, return_inverse=True)
    totals = np.bincount(group, values)
    counts = np.bincount(group)
    with np.errstate(invalid="ignore", divide="ignore"):  # alone: nan
        return (totals[group] - values) / (counts[group] - 1)


def previous_window(data: WindowDataset, values: FloatArray) -> FloatArray:
    """``values`` of the same sensor in the window before (nan if it was not there)."""
    order = np.lexsort((data.window, data.sensor_id))
    before = np.full(len(values), np.nan)
    same = (data.sensor_id[order][1:] == data.sensor_id[order][:-1]) & (
        np.diff(data.window[order]) == 1
    )
    before[order[1:][same]] = values[order[:-1][same]]
    return before


def variant(
    data: WindowDataset,
    spread: tuple[FloatArray, FloatArray],
    scaling: str = "range",
    method: str = "none",
    with_type: bool = False,
    fusion: tuple[str, ...] = (),
) -> tuple[FloatArray, tuple[str, ...]]:
    """Feature matrix and keys of one variant of the baseline features."""
    keys = PreProcess().feature_keys(method)
    columns = {key: column(data, key).copy() for key in keys}
    type_spread, sensor_spread = spread
    std = np.sqrt(column(data, "variance"))
    ranged = column(data, "has_range") == 1  # a sensor without one is z-scored already
    unit = {  # range-scaled values are multiplied by this
        "range": np.ones(len(std)),
        "type noise": 1 / type_spread,
        "sensor noise": 1 / sensor_spread,
        "window z-score": np.divide(1, std, out=np.zeros(len(std)), where=std > 0),
    }[scaling]
    unit = np.where(ranged, unit, 1.0)
    for key, power in UNIT_POWER.items():
        if key in columns:
            columns[key] *= unit**power
    if scaling == "window z-score":  # the no-range path of PreProcess
        columns["has_range"] = np.zeros(len(std))
        columns["mean"] = np.where(ranged, 0.0, columns["mean"])
        columns["spatial_delta_mean"] = np.where(
            ranged, 0.0, columns["spatial_delta_mean"]
        )
        delta_last = column(data, "z_last") - column(data, "nbr_z_last")
        columns["spatial_delta_last"] = np.where(
            ranged, np.nan_to_num(delta_last), columns["spatial_delta_last"]
        )
    if with_type:
        columns["type"] = np.array([TYPE_CODES[name] for name in data.sensor_type])
    noise = std / type_spread  # fusion compares spreads in each type's own noise unit
    steep = np.abs(column(data, "slope")) / type_spread
    if "nbr" in fusion:
        columns["nbr_std_ratio"] = ratio(noise, column(data, "nbr_std") / type_spread)
        columns["nbr_slope_delta"] = (
            column(data, "slope") - column(data, "nbr_slope")
        ) * unit
    if "col" in fusion:
        columns["col_std_ratio"] = ratio(noise, co_located(data, noise))
        columns["col_slope_ratio"] = ratio(steep, co_located(data, steep))
    if "delta_change" in fusion:
        delta = column(data, "spatial_delta_mean")
        columns["delta_change"] = (delta - previous_window(data, delta)) * unit
    return np.stack(list(columns.values()), axis=1), tuple(columns)

In [ ]:
BIGGER = {"n_estimators": 300, "max_depth": 8}  # XGBoost's defaults are 100 and 6
FUSION = ("nbr", "col")
# name -> variant() arguments, plus "params": XGBoost settings other than the defaults
OPTIMIZE_VARIANTS: dict[str, dict[str, Any]] = {
    "baseline": {},
    "type noise": {"scaling": "type noise"},
    "sensor noise": {"scaling": "sensor noise"},
    "window z-score": {"scaling": "window z-score"},
    **{method: {"method": method} for method in METHODS[1:]},
    **{f"{method} + type": {"method": method, "with_type": True} for method in METHODS},
    "fusion nbr": {"fusion": ("nbr",)},
    "fusion col": {"fusion": ("col",)},
    "fusion nbr + col": {"fusion": FUSION},
    "one model per type": {},  # trained per type in optimize_seed
    "type noise + fusion": {"scaling": "type noise", "fusion": FUSION},
    "type noise + type + fusion": {
        "scaling": "type noise",
        "with_type": True,
        "fusion": FUSION,
    },
    "baseline, bigger model": {"params": BIGGER},
    "none + type, bigger model": {"with_type": True, "params": BIGGER},
    "type noise + type + fusion, bigger model": {
        "scaling": "type noise",
        "with_type": True,
        "fusion": FUSION,
        "params": BIGGER,
    },
}
UNSEEN_VARIANTS = (  # also run with one type held out of training
    "baseline",
    "type noise",
    "sensor noise",
    "window z-score",
    "none + type",
    "fusion nbr + col",
    "type noise + fusion",
    "baseline, bigger model",
)
TRANSFER_VARIANTS = (  # trained on all Intel Lab types, scored on the generator
    "baseline",
    "type noise",
    "sensor noise",
    "window z-score",
    "type noise + fusion",
)
SPOOF_VARIANTS: dict[str, dict[str, Any]] = {
    "baseline": {},
    "delta_change": {"fusion": ("delta_change",)},
    "type noise + delta_change": {"scaling": "type noise", "fusion": ("delta_change",)},
}
VALIDATION_SHARE = 0.2  # the last 20 % of the training windows pick the threshold
CLASSES = get_settings().inference.classes
SPOOF_CLASSES = (*CLASSES, Anomalies.SPOOFED_DATA)
Runs = dict[str, dict[str, list[Metrics]]]  # table -> row -> one entry per run
Predict = Callable[[BoolArray, BoolArray], tuple[FloatArray, IntArray]]

In [ ]:
def class_index(data: WindowDataset, classes: tuple[Anomalies, ...]) -> IntArray:
    """0 normal, i = classes[i - 1]; spoof only counts when SPOOFED_DATA is a class."""
    label_of = {**KIND_LABELS, "spoof": Anomalies.SPOOFED_DATA}
    return np.array(
        [
            classes.index(label_of[k]) + 1 if label_of.get(k) in classes else 0
            for k in data.kind
        ]
    )


def scored(
    scores: FloatArray,
    named: IntArray,
    truth: IntArray,
    kind: npt.NDArray[np.str_],
    threshold: float = 0.5,
) -> Metrics:
    """Anomaly-or-not metrics, false alarms, F1 per anomaly class, recall of stuck."""
    flagged = scores > threshold
    predicted = np.where(flagged, named, 0)
    result = base(truth > 0, flagged) | {
        "pr_auc": float(average_precision_score(truth > 0, scores))
    }
    for index, label in enumerate(CLASSES, start=1):
        result[f"f1_{label}"] = float(
            f1_score(truth == index, predicted == index, zero_division=0)
        )
    result["recall_stuck"] = share(flagged, kind == "stuck")
    return result


def fit(
    features: FloatArray,
    truth: IntArray,
    keys: tuple[str, ...],
    classes: tuple[Anomalies, ...] = CLASSES,
    **params: Any,
) -> BaselineModel:
    """The section 4 model on these columns; ``params`` override XGBoost settings."""
    if not params:
        return BaselineModel.fit(features, truth, "none", 0.5, keys, classes)
    counts = np.bincount(truth, minlength=len(classes) + 1)
    weights = len(truth) / ((len(classes) + 1) * counts[truth])  # as BaselineModel.fit
    classifier = XGBClassifier(random_state=0, **params)
    return BaselineModel(
        classifier.fit(features, truth, sample_weight=weights),
        "none",
        0.5,
        keys,
        classes,
    )


def best_threshold(scores: FloatArray, truth: IntArray) -> float:
    """The score threshold with the highest anomaly-or-not F1 on these rows."""
    precision, recall, thresholds = precision_recall_curve(truth > 0, scores)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    return float(thresholds[np.argmax(f1[:-1])])


def late_windows(data: WindowDataset, train: BoolArray) -> BoolArray:
    """The last VALIDATION_SHARE of the ``train`` windows, cut in each dataset's own
    windows so every source adds its late windows."""
    late = np.zeros_like(train)
    for name in np.unique(data.dataset):
        rows = train & (data.dataset == name)
        if rows.any():
            late[rows] = data.window[rows] >= np.quantile(
                data.window[rows], 1 - VALIDATION_SHARE
            )
    return late


def evaluate(
    predict: Predict,
    data: WindowDataset,
    truth: IntArray,
    train: BoolArray,
    test: BoolArray,
) -> Metrics:
    """Test metrics at 0.5, and at the threshold that a model trained on the first
    windows of ``train`` finds best on the last VALIDATION_SHARE of them, cut in each
    dataset's own training windows so every source adds its late windows."""
    result = scored(*predict(train, test), truth[test], data.kind[test])
    late = late_windows(data, train)
    early = train & ~late
    scores, named = predict(early, late | test)  # one fit scores both
    on_late = late[late | test]
    threshold = best_threshold(scores[on_late], truth[late])
    picked = scored(
        scores[~on_late], named[~on_late], truth[test], data.kind[test], threshold
    )
    return result | {
        "threshold": threshold,
        **{
            f"{key} picked": picked[key]
            for key in ("f1", "recall", "false_alarms", "recall_stuck")
        },
    }

In [ ]:
def generator_table(seed: int) -> WindowDataset:
    """The section 5 generator run as a table; label = a value fault in the window."""
    telemetry, labels = generate(
        seed=seed, sensors=SENSORS, duration=DURATION, anomaly_rate=ANOMALY_RATE
    )
    readings = [Sensor.model_validate(message) for message in telemetry]
    preprocessor, decompose = PreProcess(), Decompose()
    rows, ids, numbers, faults = [], [], [], []
    for number, (window, kinds) in enumerate(sensor_windows(readings, labels)):
        table = window_table(preprocessor, decompose, "none", window)
        for sensor_id, features in table.items():
            rows.append([features[key] for key in TABLE_KEYS])
            ids.append(sensor_id)
            numbers.append(number)
            faults.append(bool(kinds[sensor_id] & set(VALUE_ANOMALIES)))
    label, nowhere = np.array(faults), np.zeros(len(rows), dtype=bool)
    nothing = np.full(len(rows), "", dtype=np.str_)
    return WindowDataset(
        features=np.array(rows, dtype=np.float64),
        label=label,
        kind=np.where(label, "value_fault", ""),
        sensor_type=np.array([sensor_id.rsplit("-", 1)[0] for sensor_id in ids]),
        real_fault=nowhere,
        event=nowhere,
        source_label=nothing,
        train=nowhere,
        out_of_range=nowhere,
        sensor_id=np.array(ids),
        window=np.array(numbers),
        dataset=np.full(len(rows), "generator"),  # dtype=np.str_ cuts it to "g"
        spreads={},
    )


def predictor(
    x: FloatArray,
    truth: IntArray,
    keys: tuple[str, ...],
    params: dict[str, Any],
    sensor_type: npt.NDArray[np.str_] | None = None,
) -> Predict:
    """Fit on some rows, score others; with ``sensor_type``, one model per type."""

    def predict(rows: BoolArray, scoring: BoolArray) -> tuple[FloatArray, IntArray]:
        if sensor_type is None:
            return fit(x[rows], truth[rows], keys, **params).predict(x[scoring])
        scores = np.zeros(int(scoring.sum()))
        named = np.zeros(int(scoring.sum()), dtype=np.int64)
        for name in np.unique(sensor_type):
            own = sensor_type == name
            model = fit(x[rows & own], truth[rows & own], keys, **params)
            scores[own[scoring]], named[own[scoring]] = model.predict(x[scoring & own])
        return scores, named

    return predict


def spoof_metrics(
    said: BoolArray, is_spoof: BoolArray, normal: BoolArray, score: FloatArray
) -> Metrics:
    """How well the windows said to be spoofed match the spoofed ones."""
    return {
        "precision": float(precision_score(is_spoof, said, zero_division=0)),
        "recall": share(said, is_spoof),
        "f1": float(f1_score(is_spoof, said, zero_division=0)),
        "normal_flagged": share(said, normal),
        "pr_auc": float(average_precision_score(is_spoof, score)),
    }


def optimize_seed(data: WindowDataset, generators: list[WindowDataset]) -> Runs:
    """Every table's rows for one injection seed."""
    runs: Runs = {}

    def add(title: str, name: str, metrics: Metrics) -> None:
        runs.setdefault(title, {}).setdefault(name, []).append(metrics)

    spread = spreads(data, data.train & (data.kind == ""))
    truth = class_index(data, CLASSES)
    usable = ~data.real_fault & ~data.event & (data.kind != "spoof")
    train, test = data.train & usable, ~data.train & usable
    for name, arguments in OPTIMIZE_VARIANTS.items():
        options = {k: v for k, v in arguments.items() if k != "params"}
        params = arguments.get("params", {})
        x, keys = variant(data, spread, **options)
        pooled = predictor(x, truth, keys, params)
        if name == "one model per type":
            predict = predictor(x, truth, keys, params, data.sensor_type)
        else:
            predict = pooled
        metrics = evaluate(predict, data, truth, train, test)
        add("All types", name, metrics)
        for held_out in TYPES if name in UNSEEN_VARIANTS else ():
            rows = test & (data.sensor_type == held_out)
            unseen = scored(
                *pooled(train & (data.sensor_type != held_out), rows),
                truth[rows],
                data.kind[rows],
            )
            add("Unseen type", name, unseen)
            add("Unseen type, per held-out type", f"{name} / {held_out}", unseen)
        if name in TRANSFER_VARIANTS:
            model = fit(x[train], truth[train], keys, **params)
            for there in generators:
                spread_there = spreads(there, np.ones(len(there.label), dtype=bool))
                scores = model.score(variant(there, spread_there, **options)[0])
                picked = base(there.label, scores > metrics["threshold"])
                add(
                    "Generator",
                    name,
                    base(there.label, scores > 0.5)
                    | {"pr_auc": float(average_precision_score(there.label, scores))}
                    | {f"{k} picked": picked[k] for k in ("recall", "false_alarms")},
                )

    spoof_truth = class_index(data, SPOOF_CLASSES)
    apart = data.real_fault | data.event
    train, test = data.train & ~apart, ~data.train & ~apart
    is_spoof, normal = spoof_truth[test] == len(SPOOF_CLASSES), spoof_truth[test] == 0
    for name, arguments in SPOOF_VARIANTS.items():
        x, keys = variant(data, spread, **arguments)
        model = fit(x[train], spoof_truth[train], keys, SPOOF_CLASSES)
        scores, named = model.predict(x[test])
        said = (scores > 0.5) & (named == len(SPOOF_CLASSES))
        spoof_score = model.classifier.predict_proba(x[test])[:, -1]
        add(
            "Spoofing",
            f"model, {name}",
            spoof_metrics(said, is_spoof, normal, spoof_score),
        )
    check, keys = StaticCheck(), PreProcess().feature_keys("none")
    said = np.array(
        [
            check.spoofed_data("held-out", dict(zip(keys, row, strict=False)))
            is Anomalies.SPOOFED_DATA
            for row in data.features[test]
        ]
    )
    add("Spoofing", "spoofed_data rule", spoof_metrics(said, is_spoof, normal, said))
    return runs


def optimize_main() -> None:
    for name in DATASETS:  # read once, so the workers do not each read the sources
        source_readings(name)
    datasets = Parallel(n_jobs=min(len(INJECTION_SEEDS), MAX_WORKERS))(
        delayed(build_dataset)("none", seed, TRAIN_KINDS, window_table, TABLE_KEYS)
        for seed in INJECTION_SEEDS
    )
    generators = [generator_table(seed) for seed in SEEDS]
    # held-out rows are not real alarms: keep them out of the logs
    logging.getLogger("guardian_layer.pipelines.static_check").setLevel(logging.ERROR)
    tables: Runs = {}
    for data in datasets:
        for title, rows in optimize_seed(data, generators).items():
            for name, metrics in rows.items():
                tables.setdefault(title, {}).setdefault(name, []).extend(metrics)
    for title, table in tables.items():
        print(f"\n## {title}\n")
        print(markdown_table({name: mean_sd(runs) for name, runs in table.items()}))

Long-running: builds the three tables in parallel worker processes (as in section 2),
then fits about 250 models one after another (about 30 minutes on 2 CPUs).

In [ ]:
optimize_main()

## 8. Short-window model

A model for the anomalies that show up inside one 20 s window (#96, part 1; the split
into a short- and a long-window model was agreed on #13). It trains on the only data
fast enough for such a window: grid frequency at 1 s and at 100 ms, and clean runs of
the generator (`mocks/telemetry_generator.py`) with faults injected like the real
sources. The kinds are those visible within seconds: `drift` is left to the
long-window model. Windows are 20 s of time (`injection.window_seconds`), as the live
buffer's, so a window holds 20 readings at 1 s and 200 at 100 ms, at their real
spacing.

Three tests, every number per source and mean ± sd over the injection seeds:

1. the held-out (last 30 %) windows of each grid source; also with the 100 ms
   timestamps cut to whole seconds, as the Twin sends them;
2. section 5 on the generator runs of `SEEDS`, with the generator's own labels;
3. the same runs through `LabelledWindows`, with the injected kinds.

The model is compared with the Intel-only model of section 4 and with the other
model families of section 3, and each source is also held out of training once. Each
source gets its own threshold, picked on the last 20 % of its training windows with
the same model (no refit: a refit moves the scores); the saved model holds a threshold
per sampling rate (1 s and 100 ms sources') and the 100 ms one as its default. The
model is
written to `SHORT_MODEL_DIR` (git-ignored); the results are in
`research/short_window/results.md`.

In [ ]:
SHORT_SOURCES = ("grid_frequency", "grid_frequency_100ms")
# drift is a long-window kind: inside 20 s it looks like normal change (#13, #96)
SHORT_KINDS = tuple(kind for kind in TRAIN_KINDS if kind != "drift")
# disjoint from section 5's SEEDS; 10 runs: without the generator the grid validation
# PR-AUC moves by less than 0.01 (+0.009, results.md section 4.1), so a smaller share
GENERATOR_TRAIN_SEEDS = tuple(range(100, 110))
GENERATOR_DURATION = DURATION  # seconds per run, as section 5: 180 windows of 20 s
# grid sources are read in block_hours blocks and cut once in time: the training
# windows in the block_hours before a later split are dropped, so no continuous
# stretch of signal lies on both sides; generator runs are independent, no gap
GAP_WINDOWS = {
    name: INJECTION.block_hours * 3600 // WINDOW_SECONDS for name in SHORT_SOURCES
}
# 20 s windows of time; the generator's period lives here only, not in the yaml
SHORT_SETTINGS = DATASET_SETTINGS.model_copy(
    update={
        "injection": INJECTION.model_copy(
            update={
                "window_seconds": float(WINDOW_SECONDS),
                "sample_seconds": {**INJECTION.sample_seconds, "generator": 1.0},
            }
        )
    }
)
SHORT_MODEL_DIR = ROOT / "models" / "short_window"
# 10 readings/s, as the Twin's live stream: the saved model keeps its threshold
LIVE_SOURCE = SHORT_SOURCES[1]

In [ ]:
BASE_KEYS = PreProcess().feature_keys("none")  # what the saved model reads
PREPROCESS = get_settings().preprocess
VALUE_KEYS = tuple(PREPROCESS.value_keys)
COUNTER_KEYS = tuple(PREPROCESS.counter_keys)
# columns that mostly name the source rather than the fault (results.md, caveats)
SOURCE_KEYS = (
    "sample_count",
    "has_range",
    "mean",
    "spatial_delta_mean",
    "spatial_delta_last",
    "neighbor_count",
)
# candidate columns: value features of the window z-scored (z_) or robust z-scored
# (rz_), shape features, decoupling at the sensor's own rate (n_); the winner's
# (results.md 4.2) are PreProcess's short-window keys, the rest research only
Z_KEYS = tuple(f"z_{key}" for key in VALUE_KEYS)
RZ_KEYS = tuple(f"rz_{key}" for key in VALUE_KEYS)
SHAPE_KEYS = PREPROCESS.shape_keys
NATIVE_KEYS = tuple(
    f"n_{key}" for method in METHODS[1:] for key in PREPROCESS.method_keys[method]
)
WIDE_KEYS = (*BASE_KEYS, *Z_KEYS, *RZ_KEYS, *SHAPE_KEYS, *NATIVE_KEYS)
# what the saved short-window model reads: the counters and PreProcess's short-window
# keys (robust z + shape + haar at the sensor's own rate)
SHORT_MODEL_KEYS = (*COUNTER_KEYS, *PreProcess().short_keys())
# bump whenever wide_window or anything it calls changes; 2: features from PreProcess
CACHE_VERSION = 2
# XGBoost settings of the short-window model, picked by tune_main (results.md 4.3):
# its defaults (100 trees, depth 6, rate 0.3) won; the best setting scored within
# 0.005 of them (300 trees, depth 8, rate 0.1: 0.924 vs 0.923) at 4x the size
SHORT_PARAMS: dict[str, Any] = {}
# readings per second -> the real source that picks its threshold (the generator
# picks none: its windows are the easy, near-repeated ones)
RATE_SOURCES = {1.0: "grid_frequency", 10.0: "grid_frequency_100ms"}
SOURCE_RATE = {"grid_frequency": 1.0, "grid_frequency_100ms": 10.0, "generator": 1.0}

In [ ]:
def wide_window(
    preprocessor: PreProcess,
    decompose: Decompose,
    method: str,
    window: list[Sensor],
    clean: str | None = None,
) -> dict[str, dict[str, float]]:
    """process_window's rows plus every candidate column (WIDE_KEYS) of each sensor:
    the experiments then choose columns instead of rebuilding windows. The candidate
    columns read each sensor's series cleaned by ``preprocessor.clean`` (as
    PreProcess.short_features does live), or, with ``clean``, by that section 10
    filter ("none": the raw series); process_window's rows stay raw."""
    rows = process_window(preprocessor, decompose, method, window)
    unique, _ = preprocessor.deduplicate(preprocessor.finite(window))
    by_sensor: dict[str, list[Sensor]] = {}
    for reading in unique:
        by_sensor.setdefault(reading.sensor_id, []).append(reading)
    for sensor_id, readings in by_sensor.items():
        times, raw = preprocessor.sensor_series(readings)
        # the registry lookup only for a named filter: section 8 runs without 10
        raw = preprocessor.clean(raw) if clean is None else CLEAN_REGISTRY[clean](raw)
        for prefix, scaled in (
            ("z_", preprocessor.normalize(raw, None)),
            ("rz_", preprocessor.robust_z(raw)),
        ):
            features = preprocessor.value_features(times, scaled, False)
            rows[sensor_id] |= {prefix + key: value for key, value in features.items()}
        rows[sensor_id] |= preprocessor.shape_features(raw)
        for decoupling in METHODS[1:]:
            rows[sensor_id] |= preprocessor.native_decoupling(
                decompose, decoupling, times, raw
            )
    return rows

In [ ]:
def generator_readings(seed: int, duration: int) -> list[LabeledReading]:
    """A clean generator run (no fault, no packet problem) as loader readings.

    Ids become "generator:<id>:<type>" like every loader's, so each sensor is its own
    station; ranges and locations are kept. The sensor without metadata is left out:
    read needs its packet_order and type.
    """
    telemetry, _ = generate(
        seed=seed, sensors=SENSORS, duration=duration, anomaly_rate=0.0
    )
    readings = []
    for message in telemetry:
        meta = message["metadata"]
        if meta is None:
            continue
        sensor_id = f"generator:{message['sensor_id']}:{meta['type']}"
        sensor = Sensor.model_validate(message | {"sensor_id": sensor_id})
        readings.append(LabeledReading(sensor))
    return readings

In [ ]:
def whole_second_window(
    preprocessor: PreProcess,
    decompose: Decompose,
    method: str,
    window: list[Sensor],
    clean: str | None = None,
) -> dict[str, dict[str, float]]:
    """wide_window on timestamps cut to whole seconds, as the Twin sends them (#13):
    the features the model would get live from a 10 readings/s stream."""
    cut = [
        reading.model_copy(
            update={"timestamp": reading.timestamp.replace(microsecond=0)}
        )
        for reading in window
    ]
    return wide_window(preprocessor, decompose, method, cut, clean)

In [ ]:
def grid_part(
    name: str,
    injection_seed: int,
    process: Process = wide_window,
    preprocess: PreprocessSettings | None = None,
) -> WindowDataset:
    """One grid source's 20 s sensor-windows (WIDE_KEYS), short-window kinds
    injected; ``preprocess``: settings other than the defaults (a cleaning setup)."""
    builder = LabelledWindows(
        SHORT_SETTINGS,
        method="none",
        preprocessor=PreProcess("none", settings=preprocess),
    )
    return builder.build(
        source_readings(name), injection_seed, SHORT_KINDS, process, WIDE_KEYS
    )


def generator_part(
    seed: int,
    injection_seed: int,
    process: Process = wide_window,
    preprocess: PreprocessSettings | None = None,
) -> WindowDataset:
    """One clean generator run's 20 s sensor-windows (WIDE_KEYS), kinds injected."""
    builder = LabelledWindows(
        SHORT_SETTINGS,
        method="none",
        preprocessor=PreProcess("none", settings=preprocess),
    )
    data = builder.read("generator", generator_readings(seed, GENERATOR_DURATION))
    return builder.build(data, injection_seed, SHORT_KINDS, process, WIDE_KEYS)

In [ ]:
@dataclasses.dataclass(frozen=True, slots=True)
class ShortTables:
    train: WindowDataset  # the grid sources and the generator training runs
    generator: WindowDataset  # the generator runs of SEEDS: never trained on
    whole_second: WindowDataset  # grid_frequency_100ms, whole-second timestamps


def short_tables(
    injection_seed: int,
    clean: str | None = None,
    preprocess: PreprocessSettings | None = None,
) -> ShortTables:
    """Every table of one injection seed, cleaned by the preprocess settings
    (``preprocess``, else the defaults) or by the section 10 filter ``clean``; the
    parts are built in parallel."""
    process, whole = wide_window, whole_second_window
    if clean is not None:
        process = partial(wide_window, clean=clean)
        whole = partial(whole_second_window, clean=clean)
    for name in SHORT_SOURCES:  # read once, so the workers do not each read them
        source_readings(name)
    runs = (*GENERATOR_TRAIN_SEEDS, *SEEDS)
    tasks = [
        *(
            delayed(grid_part)(name, injection_seed, process, preprocess)
            for name in SHORT_SOURCES
        ),
        delayed(grid_part)(SHORT_SOURCES[1], injection_seed, whole, preprocess),
        *(
            delayed(generator_part)(seed, injection_seed, process, preprocess)
            for seed in runs
        ),
    ]
    parts = Parallel(n_jobs=min(len(tasks), MAX_WORKERS))(tasks)
    grids, whole_second, generated = parts[:2], parts[2], parts[3:]
    trained = len(GENERATOR_TRAIN_SEEDS)
    return ShortTables(
        train=join_parts(
            [*grids, *generated[:trained]],
            [*SHORT_SOURCES, *(f"generator_{s}" for s in GENERATOR_TRAIN_SEEDS)],
        ),
        generator=join_parts(
            generated[trained:], [f"generator_{seed}" for seed in SEEDS]
        ),
        whole_second=whole_second,
    )

In [ ]:
def table_cache_path(
    injection_seed: int,
    clean: str | None = None,
    preprocess: PreprocessSettings | None = None,
) -> Path:
    """Where short_tables of this seed (filter, settings) are cached, keyed by
    everything they read."""
    settings = PREPROCESS if preprocess is None else preprocess
    # a section 10 filter ignores the settings' cleaning: leaving it out keeps the
    # keys of the tables built before #66 added it
    ignored = set() if clean is None else {"clean_method", "kalman"}
    filtered = (
        []
        if clean in (None, "none")
        else [[clean, CLEAN_WINDOW, EMA_ALPHA, KALMAN_PROCESS_RATIO]]
    )
    default = Path.home() / ".cache" / "guardian_layer"
    root = Path(os.environ.get("GUARDIAN_DATA_DIR", default))
    read = json.dumps(
        [
            CACHE_VERSION,
            SHORT_SETTINGS.injection.model_dump(mode="json"),
            GENERATOR_TRAIN_SEEDS,
            SEEDS,
            GENERATOR_DURATION,
            WIDE_KEYS,
            SHORT_KINDS,
            settings.model_dump(mode="json", exclude=ignored),
            *filtered,
        ],
        sort_keys=True,
    )
    key = hashlib.sha1(read.encode()).hexdigest()[:12]
    return root / "short_tables" / f"seed{injection_seed}_{key}.pkl"


def cached_short_tables(
    injection_seed: int,
    clean: str | None = None,
    preprocess: PreprocessSettings | None = None,
) -> ShortTables:
    """short_tables of this seed, from the cache when built with the same settings.
    Delete the cache folder after changing a loader or a data file; bump
    CACHE_VERSION after changing the build code: the key cannot see either."""
    path = table_cache_path(injection_seed, clean, preprocess)
    if path.exists():
        return ShortTables(**pickle.loads(path.read_bytes()))
    tables = short_tables(injection_seed, clean, preprocess)
    path.parent.mkdir(parents=True, exist_ok=True)
    parts = {
        field.name: getattr(tables, field.name) for field in dataclasses.fields(tables)
    }
    partial = path.with_name(path.name + ".part")  # an interrupted write is not cached
    partial.write_bytes(pickle.dumps(parts))
    os.replace(partial, path)
    return tables


def select(data: WindowDataset, keys: tuple[str, ...]) -> WindowDataset:
    """A wide table with only these columns, in this order."""
    columns = [WIDE_KEYS.index(key) for key in keys]
    return dataclasses.replace(data, features=data.features[:, columns])


def select_tables(tables: ShortTables, keys: tuple[str, ...]) -> ShortTables:
    return ShortTables(
        **{
            f.name: select(getattr(tables, f.name), keys)
            for f in dataclasses.fields(tables)
        }
    )

In [ ]:
# self-check (instant): the table cache key changes with what the build reads
_path = table_cache_path(0)
_kinds, SHORT_KINDS = SHORT_KINDS, SHORT_KINDS[:-1]
assert table_cache_path(0) != _path
SHORT_KINDS = _kinds
_preprocess = PREPROCESS
PREPROCESS = PREPROCESS.model_copy(update={"rolling_samples": 6})
assert table_cache_path(0) != _path
PREPROCESS = _preprocess
assert table_cache_path(0) == _path

In [ ]:
# self-check (seconds): the generator readings, the 20 s windows and the cut timestamps
assert not set(GENERATOR_TRAIN_SEEDS) & set(SEEDS), "training runs overlap section 5's"
_readings = generator_readings(0, 60)
assert all(r.sensor.metadata is not None for r in _readings)
# the sensor without metadata is left out
assert len({r.sensor.sensor_id for r in _readings}) == SENSORS - 1
assert all(r.sensor.sensor_id.startswith("generator:") for r in _readings)
_builder = LabelledWindows(SHORT_SETTINGS, method="none")
_data = _builder.read("generator", generator_readings(0, 600))
assert _builder.window_groups(_data, 1).window_slots == 20
assert "drift" not in SHORT_KINDS and set(SHORT_KINDS) < set(TRAIN_KINDS)
_window = [r.sensor for r in generator_readings(0, WINDOW_SECONDS)]
_base = process_window(PreProcess(), Decompose(), "none", _window)
# 1 s data has whole-second timestamps already: cutting must change nothing
_cut = whole_second_window(PreProcess(), Decompose(), "none", _window)
assert _cut == wide_window(PreProcess(), Decompose(), "none", _window)

# the wide table's columns (improvements): finite on flat and short windows, and the
# base columns untouched
_p = PreProcess()
assert len(_p.native_grid(np.arange(20) * 0.1, np.zeros(20))) == 20  # 10/s: 20 points
assert len(_p.native_grid(np.arange(20.0), np.zeros(20))) == 20
assert np.isfinite(_p.robust_z(np.array([1.0, 1.0, 1.0, 5.0]))).all()  # MAD 0
for _values in (np.zeros(20), np.array([1.0, 2.0]), np.arange(20.0)):
    assert np.isfinite(list(_p.shape_features(_values).values())).all()
_wide = wide_window(PreProcess(), Decompose(), "none", _window)
assert all(set(_wide[s]) == set(WIDE_KEYS) for s in _wide)
assert all(
    _wide[s][k] == _base[s][k] for s in _base for k in _base[s]
)  # base untouched
assert all(np.isfinite(list(_wide[s].values())).all() for s in _wide)
# bump CACHE_VERSION whenever wide_window or anything it calls changes

In [ ]:
def short_usable(data: WindowDataset) -> BoolArray:
    """The rows the model may see: no real fault, no event, no spoof (the static
    check owns spoofed_data)."""
    return ~data.real_fault & ~data.event & (data.kind != "spoof")


def embargo(
    data: WindowDataset,
    before: BoolArray,
    after: BoolArray,
    gaps: dict[str, int] = GAP_WINDOWS,
) -> BoolArray:
    """``before`` without its rows less than ``gaps[dataset]`` windows before the
    first ``after`` row of the same dataset.

    Window numbers grow with time (the gaps between sampled blocks included), so a
    window difference is a time difference. A dataset without a gap is kept whole,
    and so is one the gap would empty (printed).
    """
    kept = before.copy()
    for name, gap in gaps.items():
        own = data.dataset == name
        if not (after & own).any():
            continue
        first = data.window[after & own].min()
        near = before & own & (data.window >= first - gap)
        if near.sum() == (before & own).sum():
            print(f"{name}: the gap would drop every row before the cut; kept")
            continue
        kept &= ~near
    return kept


def short_train(data: WindowDataset) -> BoolArray:
    """The training rows: a grid source's early windows minus the gap before its
    test windows; every window of a generator run (its runs are tested apart)."""
    usable = short_usable(data)
    train = (data.train | (data.dataset == "generator")) & usable
    return embargo(data, train, ~data.train & usable)

In [ ]:
def short_fit(
    data: WindowDataset,
    train: BoolArray,
    params: dict[str, Any] | None = None,
    keys: tuple[str, ...] = BASE_KEYS,
) -> tuple[BaselineModel, dict[str, float], Metrics]:
    """Section 4's model on the ``train`` rows but the last VALIDATION_SHARE of each
    source (``keys``: the data's columns; ``params``: XGBoost settings); the
    best-F1 threshold on those validation rows for each source and for
    all of them pooled; and what the validation rows give at each.

    The thresholds are picked on the scores of the model returned and that model is
    not refitted: a refit moves the scores, and a 1 s threshold then misses most
    faults (research/short_window/results.md). The model keeps LIVE_SOURCE's
    threshold when it trained on that source, else the pooled one, and a threshold
    per sampling rate from RATE_SOURCES (what AnomalyDetector applies). ``params``
    default to SHORT_PARAMS.
    """
    params = SHORT_PARAMS if params is None else params
    truth = class_index(data, CLASSES)
    late = late_windows(data, train)
    early = embargo(data, train & ~late, late)
    fitted = BaselineModel.fit(
        data.features[early],
        truth[early],
        "none",
        0.5,
        keys,
        CLASSES,
        params=params,
    )
    scores = np.zeros(len(truth))
    scores[late] = fitted.score(data.features[late])
    groups = {"pooled": late} | {
        str(name): late & (data.dataset == name)
        for name in np.unique(data.dataset[late])
    }
    thresholds, info = {}, {}
    for name, rows in groups.items():
        thresholds[name] = best_threshold(scores[rows], truth[rows])
        validation = base(truth[rows] > 0, scores[rows] > thresholds[name])
        info[f"threshold_{name}"] = thresholds[name]
        info |= {f"validation_{name}_{k}": v for k, v in validation.items()}
    threshold = thresholds.get(LIVE_SOURCE, thresholds["pooled"])
    rates = {
        rate: thresholds[source]
        for rate, source in RATE_SOURCES.items()
        if source in thresholds
    }
    model = BaselineModel(
        fitted.classifier,
        "none",
        threshold,
        keys,
        CLASSES,
        rates,
        float(WINDOW_SECONDS),
    )
    return model, thresholds, info

In [ ]:
def binary_report(
    truth: BoolArray, kind: npt.NDArray[np.str_], scores: FloatArray, flagged: BoolArray
) -> Metrics:
    """Anomaly-or-not metrics (false_alarms: share of the normal rows flagged),
    PR-AUC and the recall of each injected kind the model is meant to catch."""
    result = base(truth, flagged) | {
        "pr_auc": float(average_precision_score(truth, scores))
    }
    for name in SHORT_KINDS:
        if name != "spoof":
            result[f"recall_{name}"] = share(flagged, kind == name)
    return result


def short_report(
    model: BaselineModel,
    data: WindowDataset,
    rows: BoolArray,
    threshold: float | None = None,
) -> Metrics:
    """binary_report at ``threshold`` (default: the model's), plus precision, recall
    and F1 of each anomaly class it names."""
    scores, named = model.predict(data.features[rows])
    truth = class_index(data, CLASSES)[rows]
    flagged = scores > (model.threshold if threshold is None else threshold)
    predicted = np.where(flagged, named, 0)
    result = binary_report(truth > 0, data.kind[rows], scores, flagged)
    for index, label in enumerate(CLASSES, start=1):
        is_class, said = truth == index, predicted == index
        result[f"precision_{label}"] = float(
            precision_score(is_class, said, zero_division=0)
        )
        result[f"recall_{label}"] = float(recall_score(is_class, said, zero_division=0))
        result[f"f1_{label}"] = float(f1_score(is_class, said, zero_division=0))
    return result


def spoof_rule_report(data: WindowDataset, rows: BoolArray) -> Metrics:
    """StaticCheck.spoofed_data on these rows: the share of spoof windows it flags,
    of normal windows it flags, and of rows it could not check (no range, too few
    neighbours)."""
    check, keys = StaticCheck(), PreProcess().feature_keys("none")
    results = [
        check.spoofed_data("held-out", dict(zip(keys, row, strict=True)))
        for row in data.features[rows]
    ]
    fired = np.array([result is Anomalies.SPOOFED_DATA for result in results])
    unchecked = np.array([result is CheckResult.NOT_CHECKED for result in results])
    kind = data.kind[rows]
    return {
        "recall_spoof": share(fired, kind == "spoof"),
        "normal_flagged": share(fired, kind == ""),
        "not_checked": float(unchecked.mean()),
    }


def add_run(runs: Runs, title: str, name: str, metrics: Metrics) -> None:
    runs.setdefault(title, {}).setdefault(name, []).append(metrics)


def merge_runs(into: Runs, runs: Runs) -> None:
    for title, rows in runs.items():
        for name, metrics in rows.items():
            into.setdefault(title, {}).setdefault(name, []).extend(metrics)

In [ ]:
def comparison_models() -> dict[str, BaselineModel]:
    """The current model (section 4, Intel Lab only). The #95 all-sources model was
    not kept, so it is not compared."""
    if not (MODEL_DIR / META_FILE).exists():
        sys.exit(f"no model in {MODEL_DIR}: run the baseline training section first")
    return {"Intel-only (section 4)": BaselineModel.load(MODEL_DIR)}


def test_rows(tables: ShortTables) -> dict[str, tuple[WindowDataset, BoolArray]]:
    """Test 1 (each grid source's held-out windows) and test 3 (the generator runs
    of SEEDS, every window: they never trained)."""
    data = tables.train
    held_out = ~data.train & short_usable(data)
    return {
        **{
            f"Test 1, {s}": (data, held_out & (data.dataset == s))
            for s in SHORT_SOURCES
        },
        "Test 3, generator": (tables.generator, short_usable(tables.generator)),
    }


def family_reports(tables: ShortTables, train: BoolArray) -> Runs:
    """Section 3's model families, anomaly or not, fitted once per seed on the
    training rows and scored on tests 1 and 3 together."""
    data = tables.train
    truth = class_index(data, CLASSES) > 0
    tests = test_rows(tables)
    scoring = np.concatenate([d.features[rows] for d, rows in tests.values()])
    runs: Runs = {}
    for name, (fit, seeds) in EXPLORE_MODELS.items():
        for seed in seeds:
            scores, flagged = fit(data.features[train], truth[train], scoring, seed)
            start = 0
            for title, (d, rows) in tests.items():
                end = start + int(rows.sum())
                add_run(
                    runs,
                    f"Model families: {title}",
                    name,
                    binary_report(
                        class_index(d, CLASSES)[rows] > 0,
                        d.kind[rows],
                        scores[start:end],
                        flagged[start:end],
                    ),
                )
                start = end
    return runs


def unseen_source_reports(
    tables: ShortTables,
    train: BoolArray,
    main: BaselineModel,
    thresholds: dict[str, float],
) -> dict[str, Metrics]:
    """Each source held out of training in turn (same recipe), scored on its test
    rows next to the main model on the same rows. The main model uses the source's
    own threshold; the other only has the pooled threshold of the sources it saw, as
    for a new source live."""
    data = tables.train
    tests = test_rows(tables)
    reports = {}
    for source, title in (
        *((s, f"Test 1, {s}") for s in SHORT_SOURCES),
        ("generator", "Test 3, generator"),
    ):
        unseen, others, _ = short_fit(
            data, train & (data.dataset != source), keys=main.keys
        )
        d, rows = tests[title]
        reports[f"{source}: seen"] = short_report(main, d, rows, thresholds[source])
        reports[f"{source}: unseen"] = short_report(unseen, d, rows, others["pooled"])
    return reports

In [ ]:
TWIN_SENSORS = 42  # devices in the Twin's live stream (#13)


def located(reading: Sensor, copy: int, station: int) -> Sensor:
    """A copy of the reading under a new id, at its own point (about 100 m apart)."""
    assert reading.metadata is not None  # every grid reading carries metadata
    metadata = reading.metadata.model_copy(
        update={"lat": 36.79 + 0.001 * copy, "lon": 10.18 + 0.001 * station}
    )
    return reading.model_copy(
        update={"sensor_id": f"{reading.sensor_id}#{copy}", "metadata": metadata}
    )


def latency_window() -> list[Sensor]:
    """One 20 s window of grid_frequency_100ms readings (200 per sensor), its few
    stations copied under new ids up to TWIN_SENSORS sensors: the size of a live
    window. Each copy gets its own location, so the neighbour step runs as live."""
    data = source_readings(SHORT_SOURCES[1])
    builder = LabelledWindows(SHORT_SETTINGS, method="none")
    period = float(data.info.sample_seconds)
    window = data.readings.slot // builder.slots_per_window(period)
    rows = np.flatnonzero(window == np.bincount(window).argmax())  # the fullest one
    base_time = datetime.fromtimestamp(float(data.info.start), UTC)
    readings = builder.window_sensors(
        data, rows, data.readings.value, base_time, period
    )
    stations = sorted({reading.sensor_id for reading in readings})
    copies = [
        located(reading, copy, stations.index(reading.sensor_id))
        for copy in range(-(-TWIN_SENSORS // len(stations)))
        for reading in readings
    ]
    kept = set(sorted({reading.sensor_id for reading in copies})[:TWIN_SENSORS])
    return [reading for reading in copies if reading.sensor_id in kept]


def short_latency(model: BaselineModel) -> Metrics:
    """process_window + predict_features on one live-sized window, as section 6,
    with ``model`` as the short-window model (every sensor here is fast); prints
    where one run spends its time."""
    window = latency_window()
    detector, preprocessor, decompose = (
        AnomalyDetector(model),
        PreProcess(),
        Decompose(),
    )
    seconds = []
    for run in range(WARMUP + REPEATS):
        started = perf_counter()
        detector.predict_features(
            process_window(preprocessor, decompose, "none", window)
        )
        if run >= WARMUP:
            seconds.append(perf_counter() - started)
    profiler = cProfile.Profile()
    profiler.enable()
    detector.predict_features(process_window(preprocessor, decompose, "none", window))
    profiler.disable()
    pstats.Stats(profiler).sort_stats("cumulative").print_stats(10)
    return {
        "sensors": float(len({reading.sensor_id for reading in window})),
        "readings": float(len(window)),
        "p50_ms": ms(seconds, 50),
        "p99_ms": ms(seconds, 99),
    }

In [ ]:
# self-check (seconds; reads the cached 100 ms readings): the latency window has 42
# located sensors, so the k-nearest search runs for each
_latency = latency_window()
_rows = process_window(PreProcess(), Decompose(), "none", _latency)
assert len(_rows) == TWIN_SENSORS
# located sensors get their k nearest; without locations the whole cluster (41)
_k = get_settings().preprocess.neighbors
assert all(row["neighbor_count"] == _k for row in _rows.values())

In [ ]:
def check_kinds(data: WindowDataset, rows: dict[str, BoolArray]) -> None:
    """Every kind the model learns is in each named row set of every source that has
    rows there: a time cut never leaves one side without a kind."""
    wanted = set(SHORT_KINDS) - {"spoof"}
    for part, chosen in rows.items():
        for source in np.unique(data.dataset):
            own = chosen & (data.dataset == source)
            missing = wanted - set(np.unique(data.kind[own]).tolist())
            assert not own.any() or not missing, f"{source} {part}: no {missing}"


def split_counts(data: WindowDataset, train: BoolArray) -> dict[str, dict[str, int]]:
    """Per source, the sensor-windows that train, validate, test, or the gap drops."""
    usable = short_usable(data)
    late = late_windows(data, train)
    fitted = embargo(data, train & ~late, late)
    held_out = ~data.train & usable & (data.dataset != "generator")
    dropped = (data.train | (data.dataset == "generator")) & usable & ~fitted & ~late
    return {
        str(source): {
            "training": int((fitted & own).sum()),
            "validation": int((late & own).sum()),
            "test": int((held_out & own).sum()),
            "dropped by the gap": int((dropped & own).sum()),
        }
        for source in np.unique(data.dataset)
        for own in [data.dataset == source]
    }


def short_seed(injection_seed: int) -> tuple[BaselineModel, Runs, dict[str, Any]]:
    """Tests 1 and 3, the unseen sources and the model families for one injection
    seed; the trained model; its thresholds and split counts. The short-window model
    and the families read SHORT_MODEL_KEYS, the Intel-only model and the spoof rule
    the base columns."""
    wide = cached_short_tables(injection_seed)
    tables = select_tables(wide, SHORT_MODEL_KEYS)
    base_tests = test_rows(select_tables(wide, BASE_KEYS))
    data = tables.train
    train = short_train(data)
    check_kinds(data, {"training": train, "test": ~data.train & short_usable(data)})
    short, thresholds, fitted = short_fit(data, train, keys=SHORT_MODEL_KEYS)
    others = comparison_models()
    runs: Runs = {}
    for title, (d, rows) in test_rows(tables).items():
        source = title.split(", ")[1]  # "Test 1, grid_frequency" -> grid_frequency
        for name, threshold in (
            ("short-window", thresholds[source]),  # the source's own threshold
            ("short-window, pooled threshold", thresholds["pooled"]),
            (
                "short-window, rate threshold",
                short.rate_thresholds.get(SOURCE_RATE[source], short.threshold),
            ),
            ("short-window @ 0.5", 0.5),
        ):
            add_run(runs, title, name, short_report(short, d, rows, threshold))
        base_data, base_rows = base_tests[title]  # the same rows, base columns
        for name, model in others.items():
            add_run(runs, title, name, short_report(model, base_data, base_rows))
    whole = tables.whole_second  # same windows and faults, whole-second timestamps
    add_run(
        runs,
        f"Test 1, {SHORT_SOURCES[1]}",
        "short-window, whole-second timestamps",
        short_report(
            short,
            whole,
            ~whole.train & short_usable(whole),
            thresholds[SHORT_SOURCES[1]],
        ),
    )
    generator = select(wide.generator, BASE_KEYS)
    add_run(
        runs,
        "Test 3, spoofed_data rule",
        "StaticCheck.spoofed_data",
        spoof_rule_report(generator, ~generator.real_fault & ~generator.event),
    )
    for name, metrics in unseen_source_reports(
        tables, train, short, thresholds
    ).items():
        add_run(runs, "Unseen source", name, metrics)
    merge_runs(runs, family_reports(tables, train))
    return short, runs, {"seed": injection_seed, **fitted, **split_counts(data, train)}

In [ ]:
FALSE_ALARM_BUDGET = 0.04  # share of clean generator windows (the #96 spec's 4 %)


def mean_of(runs: Runs, title: str, row: str, metric: str) -> float:
    return float(np.mean([run[metric] for run in runs[title][row]]))


def short_main() -> None:
    # held-out rows are not real alarms: keep them out of the logs
    logging.getLogger("guardian_layer.pipelines.static_check").setLevel(logging.ERROR)
    started = perf_counter()
    tables: Runs = {}
    seeds = []
    saved = None
    for injection_seed in INJECTION_SEEDS:
        model, runs, info = short_seed(injection_seed)
        merge_runs(tables, runs)
        seeds.append(info)
        if saved is None:  # the first seed's model is the one written
            saved = model
        print(f"injection seed {injection_seed} done: {json.dumps(info)}", flush=True)
    assert saved is not None
    saved.save(SHORT_MODEL_DIR)

    # test 2: section 5 on the generator's own labels, with the saved models
    for name, model in {"short-window": saved, **comparison_models()}.items():
        for seed in SEEDS:
            for title, rows in evaluate_seed(model, seed).items():
                for row, metrics in rows.items():
                    add_run(tables, f"Test 2, {title}", f"{name} / {row}", metrics)
    latency = short_latency(saved)

    clean = "Test 2, Share flagged among windows without a value fault"
    target = {
        "test 2 clean false alarms": mean_of(
            tables, clean, "short-window / clean", "model"
        ),
        "test 3 false alarms": mean_of(
            # at the 1 reading/s threshold the detector applies, not the generator's own
            tables,
            "Test 3, generator",
            "short-window, rate threshold",
            "false_alarms",
        ),
        **{
            f"{s} PR-AUC, short minus Intel-only": mean_of(
                tables, f"Test 1, {s}", "short-window", "pr_auc"
            )
            - mean_of(tables, f"Test 1, {s}", "Intel-only (section 4)", "pr_auc")
            for s in SHORT_SOURCES
        },
    }
    met = (
        target["test 2 clean false alarms"] <= FALSE_ALARM_BUDGET
        and target["test 3 false alarms"] <= FALSE_ALARM_BUDGET
        and all(value > 0 for key, value in target.items() if "PR-AUC" in key)
    )
    settings = {
        "window_seconds": WINDOW_SECONDS,
        "injection_seeds": INJECTION_SEEDS,
        "generator_training_runs": [
            GENERATOR_TRAIN_SEEDS[0],
            GENERATOR_TRAIN_SEEDS[-1],
        ],
        "generator_evaluation_runs": SEEDS,
        "generator_run_seconds": GENERATOR_DURATION,
        "gap_windows": GAP_WINDOWS,
        "kinds": SHORT_KINDS,
        "seconds": round(perf_counter() - started),
    }
    (SHORT_MODEL_DIR / "metrics.json").write_text(
        json.dumps(
            {
                "settings": settings,
                "seeds": seeds,
                "target": target | {"met": met},
                "latency": latency,
                "tables": tables,
            },
            indent=2,
        )
    )
    model_dir = SHORT_MODEL_DIR.relative_to(ROOT).as_posix()  # no local path
    print(f"model: {model_dir}; settings: {json.dumps(settings)}")
    print(f"latency, {TWIN_SENSORS} sensors x 200 readings: {json.dumps(latency)}")
    print(f"target (false alarms <= {FALSE_ALARM_BUDGET}, PR-AUC above): met={met}")
    print(json.dumps(target, indent=2))
    for title, table in tables.items():
        print(f"\n## {title}\n")
        print(markdown_table({name: mean_sd(runs) for name, runs in table.items()}))

In [ ]:
# self-check (seconds): the gap at the time cut, on one generator run named as a grid
# source; 180 windows, the first 126 train
_part = select(generator_part(100, 0), BASE_KEYS)
_grid = dataclasses.replace(_part, dataset=np.full(len(_part.kind), SHORT_SOURCES[0]))
_after = ~_grid.train
_first = int(_grid.window[_after].min())
_kept = embargo(_grid, _grid.train, _after, {SHORT_SOURCES[0]: 10})
assert not (_kept & (_grid.window >= _first - 10)).any()  # the 10 windows before go
assert (_kept == (_grid.train & (_grid.window < _first - 10))).all()  # the rest stay
assert (embargo(_grid, _grid.train, _after, {}) == _grid.train).all()  # no gap set
# a gap longer than the training windows would empty the source: kept whole
assert (
    embargo(_grid, _grid.train, _after, {SHORT_SOURCES[0]: 1000}) == _grid.train
).all()
assert not (short_usable(_part) & (_part.kind == "spoof")).any()
# a generator run trains on all its windows, a grid source on its early ones only
assert short_train(_part).sum() == short_usable(_part).sum()
assert not (short_train(_grid) & ~_grid.train).any()

# short_fit: a threshold per source and a pooled one, all on the scores of the model it
# returns; that model keeps LIVE_SOURCE's threshold when it trained on it, else the
# pooled one
_other = select(generator_part(101, 0), BASE_KEYS)
_two = join_parts(
    [_part, _other],
    ["generator_100", "generator_101"],
)
_model, _thresholds, _ = short_fit(_two, short_train(_two))
assert set(_thresholds) == {"pooled", "generator"}
assert _model.threshold == _thresholds["pooled"]
_live = dataclasses.replace(_two, dataset=np.full(len(_two.kind), LIVE_SOURCE))
_model, _thresholds, _ = short_fit(_live, _live.train & short_usable(_live))
assert set(_thresholds) == {"pooled", LIVE_SOURCE}
assert _model.threshold == _thresholds[LIVE_SOURCE]

In [ ]:
# self-check (seconds): the saved model gets a threshold per sampling rate, from
# the grid sources (two generator runs named after them here)
_grids = join_parts(
    [
        dataclasses.replace(part, dataset=np.full(len(part.kind), name))
        for part, name in zip((_part, _other), SHORT_SOURCES, strict=True)
    ],
    list(SHORT_SOURCES),
)
_model, _thresholds, _ = short_fit(_grids, short_train(_grids))
assert _model.threshold_for(20) == _thresholds[SHORT_SOURCES[0]]  # 1 reading/s
assert _model.threshold_for(200) == _thresholds[SHORT_SOURCES[1]]  # 10 readings/s
assert _model.threshold == _thresholds[LIVE_SOURCE]

In [ ]:
KEEP_MARGIN = 0.02  # most a source's in-source validation PR-AUC may drop
# sources the margin ignores: the generator is synthetic and its runs near-repeats,
# so its own in-source score does not block a variant (user decision, 2026-10-26)
KEEP_IGNORED = ("val_pr_auc_generator",)


def score_variant(
    tables: ShortTables,
    keys: tuple[str, ...],
    params: dict[str, Any] | None = None,
    with_generator: bool = True,
    unseen: bool = True,
) -> Metrics:
    """One choice of columns (and XGBoost settings), scored on validation windows
    only: each source's last training windows after the gap. In-source: the model of
    short_fit, which never trained on them. Unseen: a model trained without the
    source, on that source's same windows."""
    data = select(tables.train, keys)
    truth = class_index(data, CLASSES) > 0
    train = short_train(data)
    if not with_generator:
        train &= data.dataset != "generator"
    late = late_windows(data, train)
    sources = [str(source) for source in np.unique(data.dataset[train])]
    model, _, _ = short_fit(data, train, params, keys)
    result = {}
    for source in sources:
        rows = late & (data.dataset == source)
        scores = model.score(data.features[rows])
        result[f"val_pr_auc_{source}"] = float(
            average_precision_score(truth[rows], scores)
        )
    result["val_mean"] = float(np.mean(list(result.values())))
    if unseen and len(sources) > 1:
        for source in sources:
            other, _, _ = short_fit(
                data, train & (data.dataset != source), params, keys
            )
            rows = late & (data.dataset == source)
            scores = other.score(data.features[rows])
            result[f"unseen_pr_auc_{source}"] = float(
                average_precision_score(truth[rows], scores)
            )
        result["unseen_mean"] = float(
            np.mean([result[f"unseen_pr_auc_{source}"] for source in sources])
        )
    return result


def mean_metrics(runs: list[Metrics]) -> dict[str, float]:
    """Each metric's mean over the runs."""
    return {name: float(np.mean([run[name] for run in runs])) for name in runs[0]}


def keeps(candidate: dict[str, float], reference: dict[str, float]) -> bool:
    """The keep rule: the mean unseen-source PR-AUC rises, and no real source's
    in-source validation PR-AUC drops by more than KEEP_MARGIN (KEEP_IGNORED: not
    counted)."""
    if candidate["unseen_mean"] <= reference["unseen_mean"]:
        return False
    return all(
        candidate[name] >= reference[name] - KEEP_MARGIN
        for name in reference
        if name.startswith("val_pr_auc_") and name not in KEEP_IGNORED
    )


def experiment(
    title: str, variants: dict[str, dict[str, Any]]
) -> dict[str, list[Metrics]]:
    """Every variant (score_variant arguments) on each injection seed's cached
    tables; prints the mean ± sd table and returns the runs per variant."""
    runs: Runs = {}
    for injection_seed in INJECTION_SEEDS:
        tables = cached_short_tables(injection_seed)
        for name, arguments in variants.items():
            add_run(runs, title, name, score_variant(tables, **arguments))
        print(f"{title}: injection seed {injection_seed} done", flush=True)
    print(f"\n## {title}\n")
    cells = {name: mean_sd(rows) for name, rows in runs[title].items()}
    # markdown_table takes its columns from the first row: fill the short rows
    columns = max(cells.values(), key=len)
    print(
        markdown_table(
            {name: dict.fromkeys(columns, "") | row for name, row in cells.items()}
        )
    )
    return runs[title]

In [ ]:
def generator_main() -> None:
    """Does the generator help the grid sources? The grid validation PR-AUC with and
    without it, and what the generator rule makes of the change."""
    variants = {
        "all sources": {"keys": BASE_KEYS, "unseen": False},
        "without the generator": {
            "keys": BASE_KEYS,
            "with_generator": False,
            "unseen": False,
        },
    }
    runs = experiment("Generator experiment", variants)
    grid = [f"val_pr_auc_{source}" for source in SHORT_SOURCES]
    with_it, without = (
        float(np.mean([mean_metrics(runs[name])[key] for key in grid]))
        for name in variants
    )
    change = without - with_it
    if change <= -0.01:
        verdict = "keep all 50 runs"
    elif change < 0.01:
        verdict = "keep 10 runs"
    else:
        verdict = "drop the generator from training"
    print(f"grid validation PR-AUC without the generator: {change:+.4f} -> {verdict}")

In [ ]:
# self-check (instant): the keep rule of the improvement experiments
_reference = {"unseen_mean": 0.2, "val_pr_auc_a": 0.80}
assert keeps({"unseen_mean": 0.3, "val_pr_auc_a": 0.79}, _reference)
assert not keeps({"unseen_mean": 0.3, "val_pr_auc_a": 0.77}, _reference)  # in-source
assert not keeps({"unseen_mean": 0.2, "val_pr_auc_a": 0.80}, _reference)  # no gain
# the generator's own in-source score does not count (synthetic, near-repeated runs)
_with_generator = _reference | {"val_pr_auc_generator": 0.99}
assert keeps(
    {"unseen_mean": 0.3, "val_pr_auc_a": 0.80, "val_pr_auc_generator": 0.5},
    _with_generator,
)

In [ ]:
# column choices of the generalization experiments (research only: the saved model
# keeps PreProcess's features)
NORMALIZATIONS = {
    "base": BASE_KEYS,
    "base, no source columns": tuple(
        key for key in BASE_KEYS if key not in SOURCE_KEYS
    ),
    "z-score": (
        *COUNTER_KEYS,
        *(
            key
            for key in Z_KEYS
            if key not in ("z_sample_count", "z_has_range", "z_mean")
        ),
    ),
    "robust z": (
        *COUNTER_KEYS,
        *(
            key
            for key in RZ_KEYS
            if key not in ("rz_sample_count", "rz_has_range", "rz_mean")
        ),
    ),
}
DECOUPLING_GROUPS = {
    method: tuple(f"n_{key}" for key in PREPROCESS.method_keys[method])
    for method in METHODS[1:]
}


def best_variant(
    runs: dict[str, list[Metrics]], reference: str, candidates: tuple[str, ...]
) -> str:
    """The candidate that keeps against the reference with the highest mean
    unseen-source PR-AUC; the reference when none keeps."""
    base = mean_metrics(runs[reference])
    best, best_unseen = reference, base["unseen_mean"]
    for name in candidates:
        metrics = mean_metrics(runs[name])
        if keeps(metrics, base) and metrics["unseen_mean"] > best_unseen:
            best, best_unseen = name, metrics["unseen_mean"]
    return best


def generalization_main() -> None:
    """Stage A picks a normalization, stage B adds shape features and each
    decoupling method to it; every pick by the keep rule on validation windows."""
    stage_a = experiment(
        "Generalization, stage A: normalization",
        {name: {"keys": keys} for name, keys in NORMALIZATIONS.items()},
    )
    normal = best_variant(stage_a, "base", tuple(NORMALIZATIONS)[1:])
    keys = NORMALIZATIONS[normal]
    variants = {
        normal: {"keys": keys},
        f"{normal} + shape": {"keys": (*keys, *SHAPE_KEYS)},
        **{
            f"{normal} + {method}": {"keys": (*keys, *group)}
            for method, group in DECOUPLING_GROUPS.items()
        },
    }
    stage_b = experiment("Generalization, stage B: shape and decoupling", variants)
    shape = f"{normal} + shape"
    shape_kept = keeps(mean_metrics(stage_b[shape]), mean_metrics(stage_b[normal]))
    methods = tuple(f"{normal} + {method}" for method in DECOUPLING_GROUPS)
    decoupling = best_variant(stage_b, normal, methods)
    winner = best_variant(stage_b, normal, (shape, *methods))
    if shape_kept and decoupling != normal:  # both help alone: try them together
        method = decoupling.removeprefix(f"{normal} + ")
        both = f"{normal} + shape + {method}"
        stage_c = experiment(
            "Generalization, stage C: together",
            {
                normal: {"keys": keys},
                both: {"keys": (*keys, *SHAPE_KEYS, *DECOUPLING_GROUPS[method])},
            },
        )
        if (
            best_variant(stage_c, normal, (both,)) == both
            and mean_metrics(stage_c[both])["unseen_mean"]
            > mean_metrics(stage_b[winner])["unseen_mean"]
        ):
            winner = both
    print(
        f"stage A: {normal}; stage B: shape kept {shape_kept}, decoupling {decoupling}"
    )
    print(f"winner: {winner}")

In [ ]:
TUNING_GRID = tuple(
    {"n_estimators": trees, "max_depth": depth, "learning_rate": rate}
    for trees in (100, 300)
    for depth in (4, 6, 8)
    for rate in (0.1, 0.3)
)  # XGBoost's defaults (100, 6, 0.3) included


def tuning_name(params: dict[str, Any]) -> str:
    return (
        f"{params['n_estimators']} trees, depth {params['max_depth']}, "
        f"rate {params['learning_rate']}"
    )


def tune_main() -> None:
    """Each XGBoost setting on the validation windows; the pick is the highest mean
    in-source validation PR-AUC, or, within 0.005 of it, the smallest model
    (trees x depth: what inference costs)."""
    variants = {
        tuning_name(params): {"keys": BASE_KEYS, "params": params, "unseen": False}
        for params in TUNING_GRID
    }
    runs = experiment("Tuning", variants)
    score = {name: mean_metrics(rows)["val_mean"] for name, rows in runs.items()}
    close = [
        params
        for params in TUNING_GRID
        if score[tuning_name(params)] >= max(score.values()) - 0.005
    ]
    sizes = [params["n_estimators"] * params["max_depth"] for params in close]
    pick = close[sizes.index(min(sizes))]
    print(f"pick: {tuning_name(pick)} -> SHORT_PARAMS = {pick}")

In [ ]:
# self-check (instant): every variant's columns exist in the wide table, once each
for _keys in (*NORMALIZATIONS.values(), *DECOUPLING_GROUPS.values(), SHAPE_KEYS):
    assert set(_keys) <= set(WIDE_KEYS) and len(set(_keys)) == len(_keys)
assert not set(SOURCE_KEYS) & set(NORMALIZATIONS["base, no source columns"])
assert SHORT_MODEL_KEYS == (
    *NORMALIZATIONS["robust z"],
    *SHAPE_KEYS,
    *DECOUPLING_GROUPS["haar"],
)

Long-running: per injection seed, 53 generator runs and three grid builds in parallel worker processes, then about 30 model fits; the first run also reads both grid sources (slow, then cached). Needs the model from section 4.

In [ ]:
short_main()

## 9. Long-window model

#96 part 2: drift, stuck values and fake signals on slow sensors, judged on 6 h and 24 h
windows. Trained and tested on the 19 slow city sources only (no grid, no generator,
never the Twin). Each sensor's readings are averaged onto a time grid
(`pipelines/long_preprocess.py`); `process_long_window` builds the `lw_` rows, and the
model is section 4's XGBoost on them. The same build also writes section 4's columns, so
the Intel-only model is compared on the same windows. Results:
`research/long_window/results.md`; models: `models/long_window/6h/` and `24h/`
(git-ignored), each with `metrics.json`.

In [ ]:
LONG_SOURCES = (
    "array_of_things",
    "beijing",
    "bellinge",
    "chicago_beach",
    "emergency_calls",
    "ett",
    "intel_lab",
    "jena",
    "melbourne",
    "nab",
    "noaa_tides",
    "sensor_community",
    "traffic_speeds",
    "uci_air_quality",
    "uci_household",
    "uci_load",
    "uk_flood",
    "uk_traffic",
    "usgs",
)
LONG_WINDOW = get_settings().long_window
LONG_KEYS = get_settings().preprocess.long_keys
LONG_KINDS = (
    *DATASET_SETTINGS.long_injection.fault_kinds,
    *DATASET_SETTINGS.long_injection.fabricated_kinds,
)
# the long-only features, each dropped in turn (the neighbour count goes with its delta)
NEW_FEATURES = {
    "trend vs scatter": ("lw_trend_scatter_ratio",),
    "flat run": ("lw_flat_run_share",),
    "neighbour slope": ("lw_neighbour_slope_delta", "lw_neighbor_count"),
}
REUSE_KEYS = tuple(
    key for key in LONG_KEYS if not any(key in keys for keys in NEW_FEATURES.values())
)
# one build, two column sets: the long rows, then section 4's (sample_count among them)
LONG_COLUMNS = (*LONG_KEYS, *BASE_KEYS)
LONG_MODEL_DIR = ROOT / "models" / "long_window"
FALSE_ALARMS_PER_SENSOR_DAY = 1 / 25  # the spec's budget, the same for 6 h and 24 h
LONG_WORKERS = 6  # bellinge and uci_load hold about 2 GB of readings each
# the longest builds (most readings), started first so they do not start last
LONG_FIRST = ("uci_load", "bellinge", "array_of_things", "melbourne", "uci_household")
LONG_CACHE_VERSION = 2  # bump after changing the build code: the key cannot see it
LIVE_HOURS = 24  # test 5: hours of each source cut into live 20 s windows

In [ ]:
def window_name(window_seconds: float) -> str:
    return f"{int(window_seconds // 3600)}h"


def long_source(name: str) -> SourceReadings:
    """The source read for the long windows: only Array of Things is read another way
    (24 h blocks, every 1,801 h); the others reuse section 1's file."""
    settings = DATASET_SETTINGS.long_window(LONG_WINDOW.window_seconds[0])
    sampled = read_key(name, settings) != read_key(name, DATASET_SETTINGS)
    return source_readings(name, settings, ".long" if sampled else "")


def long_and_base(
    long_preprocess: LongPreProcess,
    preprocessor: PreProcess,
    decompose: Decompose,
    method: str,
    window: list[Sensor],
) -> dict[str, dict[str, float]]:
    """process_long_window's rows and process_window's, of the same window."""
    base_rows = process_window(preprocessor, decompose, method, window)
    long_rows = process_long_window(
        long_preprocess, preprocessor, decompose, method, window
    )
    return {
        sensor_id: base_rows[sensor_id] | row for sensor_id, row in long_rows.items()
    }


def long_part(
    name: str,
    window_seconds: float,
    injection_seed: int,
    long_window: LongWindowSettings = LONG_WINDOW,
) -> WindowDataset | None:
    """One source's long windows, faults injected with this seed, built with these
    long-window settings (their cleaning: section 12); None when it has no clean
    window to inject into (printed)."""
    # one line per window from process_long_window: keep them out; process_window
    # (section 4's columns) drops a long window's far packet orders with a warning
    # each, its 20 s behaviour kept as it is for the Intel-only comparison
    logging.getLogger("guardian_layer.pipelines.long_preprocess").setLevel(
        logging.WARNING
    )
    logging.getLogger("guardian_layer.pipelines.preprocess").setLevel(logging.ERROR)
    long_preprocess = LongPreProcess(PreProcess(), Decompose(), long_window)
    builder = LabelledWindows(
        DATASET_SETTINGS.long_window(window_seconds), method="none"
    )
    try:
        return builder.build(
            long_source(name),
            injection_seed,
            LONG_KINDS,
            partial(long_and_base, long_preprocess),
            LONG_COLUMNS,
        )
    except ValueError as error:  # no clean sensor-window with min_samples readings
        print(f"{name} {window_name(window_seconds)}: skipped: {error}", flush=True)
        return None


def long_table_path(
    window_seconds: float,
    injection_seed: int,
    sources: tuple[str, ...],
    long_window: LongWindowSettings = LONG_WINDOW,
) -> Path:
    """Where long_table is cached, keyed by everything it reads."""
    default = Path.home() / ".cache" / "guardian_layer"
    root = Path(os.environ.get("GUARDIAN_DATA_DIR", default))
    read = json.dumps(
        [
            LONG_CACHE_VERSION,
            DATASET_SETTINGS.long_window(window_seconds).model_dump(mode="json"),
            long_window.model_dump(mode="json"),
            get_settings().preprocess.model_dump(mode="json"),
            sources,
            LONG_KINDS,
            LONG_COLUMNS,
        ],
        sort_keys=True,
    )
    key = hashlib.sha1(read.encode()).hexdigest()[:12]
    name = f"{window_name(window_seconds)}_seed{injection_seed}_{key}.pkl"
    return root / "long_tables" / name


def long_table(
    window_seconds: float,
    injection_seed: int,
    sources: tuple[str, ...] = LONG_SOURCES,
    long_window: LongWindowSettings = LONG_WINDOW,
) -> WindowDataset:
    """Every source's long windows of this seed (and long-window settings), joined;
    built in parallel, cached."""
    path = long_table_path(window_seconds, injection_seed, sources, long_window)
    if path.exists():
        return WindowDataset(**pickle.loads(path.read_bytes()))
    for name in sources:  # read once here, so the workers do not each read them
        long_source(name)
    order = [
        *(name for name in sources if name in LONG_FIRST),
        *(name for name in sources if name not in LONG_FIRST),
    ]
    parts = Parallel(n_jobs=min(len(sources), LONG_WORKERS))(
        delayed(long_part)(name, window_seconds, injection_seed, long_window)
        for name in order
    )
    built = dict(zip(order, parts, strict=True))
    kept = [name for name in sources if built[name] is not None]
    table = join_parts([built[name] for name in kept], kept)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = {f.name: getattr(table, f.name) for f in dataclasses.fields(table)}
    # an interrupted write is not cached
    partial_path = path.with_name(path.name + ".part")
    partial_path.write_bytes(pickle.dumps(fields))
    os.replace(partial_path, path)
    return table

In [ ]:
def long_column(data: WindowDataset, key: str) -> FloatArray:
    return data.features[:, LONG_COLUMNS.index(key)]


def long_scorable(data: WindowDataset) -> BoolArray:
    """The rows with enough filled grid buckets: LongAnomalyDetector leaves the
    others out."""
    return long_column(data, "lw_grid_points") >= LONG_WINDOW.min_grid_points


def long_usable(data: WindowDataset) -> BoolArray:
    """The rows a long model may see: scorable, no real fault and no event."""
    return ~data.real_fault & ~data.event & long_scorable(data)


def long_splits(data: WindowDataset) -> tuple[BoolArray, BoolArray, BoolArray]:
    """Fit, validation and test rows. Per source by time: the first train_share
    trains, of which the last VALIDATION_SHARE validates (it picks the thresholds);
    one window of gap before each later part, so no stretch of signal lies on both
    sides."""
    usable = long_usable(data)
    gaps = {str(name): 1 for name in np.unique(data.dataset)}
    test = ~data.train & usable
    train = embargo(data, data.train & usable, test, gaps)
    late = late_windows(data, train)
    early = embargo(data, train & ~late, late, gaps)
    return early, late, test


def per_sensor_day(
    flagged: BoolArray, clean: BoolArray, window_seconds: float
) -> float:
    """Flagged clean windows per sensor-day."""
    days = clean.sum() * window_seconds / 86400
    return float(flagged[clean].sum() / days) if days else float("nan")


def budget_threshold(
    scores: FloatArray, truth: IntArray, window_seconds: float
) -> float:
    """The best-F1 threshold (flag: score > threshold) among those that keep the clean
    rows within FALSE_ALARMS_PER_SENSOR_DAY."""
    positive = truth > 0
    clean = np.sort(scores[~positive])[::-1]
    allowed = int(FALSE_ALARMS_PER_SENSOR_DAY * len(clean) * window_seconds / 86400)
    # the lowest threshold within budget: at most ``allowed`` clean scores above it
    floor = float(clean[allowed]) if allowed < len(clean) else -np.inf
    hits = np.sort(scores[positive])
    candidates = np.unique(
        np.concatenate(([floor], np.nextafter(hits[hits > floor], -np.inf)))
    )
    candidates = candidates[candidates >= floor]
    caught = len(hits) - np.searchsorted(hits, candidates, side="right")
    raised = len(clean) - np.searchsorted(clean[::-1], candidates, side="right")
    f1 = 2 * caught / np.maximum(2 * caught + raised + (len(hits) - caught), 1)
    return float(candidates[np.argmax(f1)])


def sample_rate(data: WindowDataset) -> FloatArray:
    """Readings per second each row's source sends (the rate thresholds' key)."""
    seconds = {
        str(name): INJECTION.sample_seconds[str(name)]
        for name in np.unique(data.dataset)
    }
    return np.array([1.0 / seconds[str(name)] for name in data.dataset])


def long_fit(
    data: WindowDataset,
    early: BoolArray,
    late: BoolArray,
    keys: tuple[str, ...],
    window_seconds: float,
) -> tuple[BaselineModel, Metrics]:
    """Section 4's model on these columns of the ``early`` rows, with the budget
    threshold of the ``late`` rows pooled, and one per sampling rate when the pooled
    one keeps a rate over budget."""
    columns = [LONG_COLUMNS.index(key) for key in keys]
    truth = class_index(data, CLASSES)
    fitted = BaselineModel.fit(
        data.features[early][:, columns], truth[early], "haar", 0.5, keys, CLASSES
    )
    scores = fitted.score(data.features[late][:, columns])
    pooled = budget_threshold(scores, truth[late], window_seconds)
    rate = sample_rate(data)[late]
    over = [
        float(r)
        for r in np.unique(rate)
        if per_sensor_day(
            scores > pooled, (truth[late] == 0) & (rate == r), window_seconds
        )
        > FALSE_ALARMS_PER_SENSOR_DAY
    ]
    rates = {}
    if over:
        rates = {
            float(r): budget_threshold(
                scores[rate == r], truth[late][rate == r], window_seconds
            )
            for r in np.unique(rate)
        }
    model = BaselineModel(
        fitted.classifier, "haar", pooled, keys, CLASSES, rates, float(window_seconds)
    )
    return model, {"threshold": pooled, "rates_over_budget": float(len(over))}


def long_scores(
    model: BaselineModel, data: WindowDataset, rows: BoolArray
) -> tuple[FloatArray, BoolArray]:
    """Scores and flags of these rows, each at the threshold of its reading count, as
    the detector applies them."""
    columns = [LONG_COLUMNS.index(key) for key in model.keys]
    scores = model.score(data.features[rows][:, columns])
    counts = long_column(data, "sample_count")[rows]
    thresholds = np.array([model.threshold_for(count) for count in counts])
    return scores, scores > thresholds


def long_report(
    truth: IntArray,
    kind: npt.NDArray[np.str_],
    scores: FloatArray,
    flagged: BoolArray,
    window_seconds: float,
) -> Metrics:
    """Anomaly-or-not metrics, false alarms per sensor-day, recall of each kind."""
    positive = truth > 0
    result = base(positive, flagged) | {
        "pr_auc": float(average_precision_score(positive, scores))
        if positive.any() and (~positive).any()
        else float("nan"),
        "false_alarms_per_sensor_day": per_sensor_day(
            flagged, ~positive, window_seconds
        ),
    }
    for name in LONG_KINDS:
        result[f"recall_{name}"] = share(flagged, kind == name)
    return result

In [ ]:
def live_part(name: str) -> tuple[FloatArray, BoolArray] | None:
    """Test 5: LIVE_HOURS of the source (from its time cut, section 1's read) in live
    20 s windows, as the former detector sent slow sensors to the Intel-only model:
    section 4's columns per sensor-window, and which windows are clean. No faults are
    injected: a window holds 0-1 readings, too few to place one. None when no window
    is left (printed)."""
    data = source_readings(name)
    period = float(data.info.sample_seconds)
    start = float(np.quantile(data.readings.slot, INJECTION.train_share))
    kept = (data.readings.slot >= start) & (
        data.readings.slot < start + LIVE_HOURS * 3600 / period
    )
    readings = ReadingColumns(
        **{
            f.name: getattr(data.readings, f.name)[kept]
            for f in dataclasses.fields(ReadingColumns)
        }
    )
    sliced = dataclasses.replace(data, readings=readings)
    live = DATASET_SETTINGS.model_copy(
        update={"injection": INJECTION.model_copy(update={"window_seconds": 20.0})}
    )
    builder = LabelledWindows(live, method="none")
    try:
        groups = builder.window_groups(sliced, 1)  # one reading is a window here
    except ValueError as error:
        print(f"{name} live: skipped: {error}", flush=True)
        return None
    rows, out = builder.window_features(
        sliced, groups, sliced.readings.value, process_window, BASE_KEYS
    )
    return np.array(rows, dtype=np.float64), groups.clean[out]


def live_reports(sources: tuple[str, ...]) -> Runs:
    """Test 5: the Intel-only model on live-shaped windows of each slow source."""
    intel = BaselineModel.load(MODEL_DIR)
    parts = Parallel(n_jobs=min(len(sources), LONG_WORKERS))(
        delayed(live_part)(name) for name in sources
    )
    runs: Runs = {}
    count = BASE_KEYS.index("sample_count")
    for name, part in zip(sources, parts, strict=True):
        if part is None:
            continue
        features, clean = part
        flagged = intel.score(features) > intel.threshold
        add_run(
            runs,
            "Test 5, Intel-only on live 20 s windows",
            name,
            {
                "windows": float(clean.sum()),
                "readings_per_window": float(features[clean, count].mean()),
                f"share_with_{MIN_SAMPLES}_or_more": share(
                    features[:, count] >= MIN_SAMPLES, clean
                ),
                "false_alarms": share(flagged, clean),
            },
        )
    return runs

In [ ]:
def held_out_reports(
    data: WindowDataset,
    test: BoolArray,
    models: dict[str, BaselineModel],
    window_seconds: float,
) -> Runs:
    """Tests 1-3 and the comparisons on the held-out rows of one table."""
    truth = class_index(data, CLASSES)
    usable = long_scorable(data)  # the rows LongAnomalyDetector scores
    runs: Runs = {}
    scored = {
        name: long_scores(model, data, np.ones(len(truth), bool))
        for name, model in models.items()
    }
    for name, (scores, flagged) in scored.items():
        add_run(
            runs,
            "Comparisons, all held-out windows",
            name,
            long_report(
                truth[test],
                data.kind[test],
                scores[test],
                flagged[test],
                window_seconds,
            ),
        )
    scores, flagged = scored["long-window"]
    for source in np.unique(data.dataset):
        own = data.dataset == source
        rows = test & own
        if rows.any():  # test 1
            add_run(
                runs,
                "Test 1, held-out windows per source",
                str(source),
                long_report(
                    truth[rows],
                    data.kind[rows],
                    scores[rows],
                    flagged[rows],
                    window_seconds,
                ),
            )
        clean = rows & (truth == 0)
        for title, special in (
            ("Test 2, real faults", data.real_fault & own & usable),
            ("Test 3, events", data.event & own & usable),
        ):
            if special.any():
                for name, (_, flags) in scored.items():
                    if name in ("long-window", "Intel-only (section 4)"):
                        add_run(
                            runs,
                            title,
                            f"{source} / {name}",
                            {
                                "windows": float(special.sum()),
                                "flagged": share(flags, special),
                                "clean_test_flagged": share(flags, clean),
                            },
                        )
    return runs


def family_long_reports(
    data: WindowDataset, early: BoolArray, test: BoolArray, window_seconds: float
) -> Runs:
    """Section 3's model families on the long columns, fitted on the fit rows."""
    truth = class_index(data, CLASSES)
    columns = [LONG_COLUMNS.index(key) for key in LONG_KEYS]
    features = data.features[:, columns]
    runs: Runs = {}
    for name, (fit_family, seeds) in EXPLORE_MODELS.items():
        for seed in seeds:
            scores, flagged = fit_family(
                features[early], truth[early] > 0, features[test], seed
            )
            add_run(
                runs,
                "Comparisons, all held-out windows",
                f"family: {name}",
                long_report(
                    truth[test], data.kind[test], scores, flagged, window_seconds
                ),
            )
    return runs


def unseen_long_reports(
    data: WindowDataset,
    early: BoolArray,
    late: BoolArray,
    test: BoolArray,
    window_seconds: float,
) -> Runs:
    """Test 4: each source left out of training in turn, scored on its held-out rows
    at the pooled threshold of the others (a new source has no rate of its own)."""
    truth = class_index(data, CLASSES)
    runs: Runs = {}
    for source in np.unique(data.dataset):
        own = data.dataset == source
        rows = test & own
        if not rows.any() or not (early & ~own).any():
            continue
        model, _ = long_fit(data, early & ~own, late & ~own, LONG_KEYS, window_seconds)
        model = BaselineModel(
            model.classifier,
            model.method,
            model.threshold,
            model.keys,
            model.classes,
            None,
            model.window_seconds,
        )
        scores, flagged = long_scores(model, data, rows)
        add_run(
            runs,
            "Test 4, unseen source",
            str(source),
            long_report(truth[rows], data.kind[rows], scores, flagged, window_seconds),
        )
    return runs


def long_latency(model: BaselineModel, window_seconds: float) -> Metrics:
    """process_long_window + LongAnomalyDetector on one Intel Lab window of this
    length, per sensor-day."""
    data = source_readings("intel_lab")
    builder = LabelledWindows(
        DATASET_SETTINGS.long_window(window_seconds), method="none"
    )
    period = float(data.info.sample_seconds)
    window = data.readings.slot // builder.slots_per_window(period)
    rows = np.flatnonzero(window == np.bincount(window).argmax())  # the fullest one
    base_time = datetime.fromtimestamp(float(data.info.start), UTC)
    readings = builder.window_sensors(
        data, rows, data.readings.value, base_time, period
    )
    long_preprocess = LongPreProcess(PreProcess(), Decompose(), LONG_WINDOW)
    detector = LongAnomalyDetector(model, LONG_WINDOW.min_grid_points)
    seconds = []
    for run in range(WARMUP + REPEATS):
        started = perf_counter()
        detector.predict_features(
            process_long_window(
                long_preprocess, PreProcess(), Decompose(), "none", readings
            )
        )
        if run >= WARMUP:
            seconds.append(perf_counter() - started)
    sensors = len({reading.sensor_id for reading in readings})
    days = sensors * window_seconds / 86400
    return {
        "sensors": float(sensors),
        "readings": float(len(readings)),
        "p50_ms": ms(seconds, 50),
        "p50_ms_per_sensor_day": ms(seconds, 50) / days,
    }

In [ ]:
def long_seed(
    window_seconds: float, injection_seed: int, sources: tuple[str, ...]
) -> tuple[BaselineModel, Runs, Metrics]:
    """One window length, one injection seed: the model and every table."""
    data = long_table(window_seconds, injection_seed, sources)
    early, late, test = long_splits(data)
    main, info = long_fit(data, early, late, LONG_KEYS, window_seconds)
    models = {"long-window": main}
    models["reuse only"] = long_fit(data, early, late, REUSE_KEYS, window_seconds)[0]
    for name, keys in NEW_FEATURES.items():
        kept = tuple(key for key in LONG_KEYS if key not in keys)
        models[f"without {name}"] = long_fit(data, early, late, kept, window_seconds)[0]
    models["Intel-only (section 4)"] = BaselineModel.load(MODEL_DIR)
    runs = held_out_reports(data, test, models, window_seconds)
    merge_runs(runs, family_long_reports(data, early, test, window_seconds))
    merge_runs(runs, unseen_long_reports(data, early, late, test, window_seconds))
    info |= {
        "rows": float(len(data.kind)),
        "fit_rows": float(early.sum()),
        "validation_rows": float(late.sum()),
        "test_rows": float(test.sum()),
        "sources": float(len(np.unique(data.dataset))),
    }
    return main, runs, info


def long_main(
    sources: tuple[str, ...] = LONG_SOURCES,
    seeds: tuple[int, ...] = INJECTION_SEEDS,
    windows: tuple[float, ...] = LONG_WINDOW.window_seconds,
) -> None:
    live = live_reports(sources)  # test 5: no seed, no window length
    for window_seconds in windows:
        started = perf_counter()  # this length's run, test 5 not included
        tables: Runs = {}
        merge_runs(tables, live)
        seeds_info = []
        saved = None
        for injection_seed in seeds:
            model, runs, info = long_seed(window_seconds, injection_seed, sources)
            merge_runs(tables, runs)
            seeds_info.append(info)
            if saved is None:  # the first seed's model is the one written
                saved = model
            name = window_name(window_seconds)
            print(f"{name} seed {injection_seed} done: {json.dumps(info)}", flush=True)
        assert saved is not None
        model_dir = LONG_MODEL_DIR / window_name(window_seconds)
        saved.save(model_dir)
        latency = long_latency(saved, window_seconds)
        pooled = "Comparisons, all held-out windows"
        target = {
            "clean false alarms per sensor-day": mean_of(
                tables, pooled, "long-window", "false_alarms_per_sensor_day"
            ),
            "PR-AUC minus reuse only": mean_of(tables, pooled, "long-window", "pr_auc")
            - mean_of(tables, pooled, "reuse only", "pr_auc"),
            "PR-AUC minus Intel-only": mean_of(tables, pooled, "long-window", "pr_auc")
            - mean_of(tables, pooled, "Intel-only (section 4)", "pr_auc"),
        }
        met = (
            target["clean false alarms per sensor-day"] <= FALSE_ALARMS_PER_SENSOR_DAY
            and target["PR-AUC minus reuse only"] > 0
            and target["PR-AUC minus Intel-only"] > 0
        )
        settings = {
            "window_seconds": window_seconds,
            "grid_seconds": LONG_WINDOW.grid_seconds,
            "min_grid_points": LONG_WINDOW.min_grid_points,
            "injection_seeds": seeds,
            "sources": sources,
            "kinds": LONG_KINDS,
            "budget_per_sensor_day": FALSE_ALARMS_PER_SENSOR_DAY,
            # a fault is seen once its window closes and the next job runs
            "detection_delay_hours_at_most": (
                f"{window_seconds / 3600:g} + the job interval"
            ),
            "seconds": round(perf_counter() - started),
        }
        (model_dir / "metrics.json").write_text(
            json.dumps(
                {
                    "settings": settings,
                    "seeds": seeds_info,
                    "target": target | {"met": met},
                    "latency": latency,
                    "tables": tables,
                },
                indent=2,
            )
        )
        relative = model_dir.relative_to(ROOT).as_posix()  # no local path
        print(f"\n# {window_name(window_seconds)}: model {relative}")
        print(f"settings: {json.dumps(settings)}")
        print(f"latency: {json.dumps(latency)}")
        budget = FALSE_ALARMS_PER_SENSOR_DAY
        print(f"target (<= {budget} per sensor-day, PR-AUC above): met={met}")
        print(json.dumps(target, indent=2))
        for title, table in tables.items():
            print(f"\n## {title}\n")
            print(markdown_table({name: mean_sd(runs) for name, runs in table.items()}))

In [ ]:
# self-check (instant): the budget threshold keeps clean false alarms within budget
# and picks the best F1 above that
_scores = np.array([0.1, 0.2, 0.3, 0.9, 0.4, 0.8, 0.95])
_truth = np.array([0, 0, 0, 0, 1, 1, 1])
# 4 clean 1-day windows allow 0.16 false alarms: none, so the threshold is >= 0.9
_threshold = budget_threshold(_scores, _truth, 86400.0)
assert _threshold >= 0.9 and (_scores[_truth == 0] > _threshold).sum() == 0
assert (_scores[_truth > 0] > _threshold).sum() == 1  # only 0.95 stays above
# 100 clean days allow 4: the 0.9 clean score may then be flagged
_threshold = budget_threshold(_scores, _truth, 100 * 86400.0)
assert _threshold < 0.4  # every anomaly caught, 1 of 4 allowed false alarms used

Long-running: for each window length and injection seed, the 19 sources in up to 4 worker processes, then about 30 model fits; the first run also reads Array of Things in 24 h blocks (hours, then cached). Needs the model from section 4.

In [ ]:
long_main()

## 10. Noise cleaning

#66 asked for a noise-cleaning step before the features, so that noise is removed
"without removing the signals the model has to detect". After the split into a short-
and a long-window model (#96), the lead agreed on #66 to: "test two or three filters
against the short model, comparing false positives and per-kind recall. Keep a
cleaning step only if false positives drop and recall doesn't. Otherwise close #66
with the numbers."

The three filters of the issue, one setting each: a centred moving median, an
exponential moving average (a first-order low-pass) and a 1-D Kalman filter. A filter
replaces each sensor's raw series before the short-window columns are computed
(section 8's `wide_window`); the base columns and packet counters stay on the raw
readings. For each filter and injection seed the short-window model is retrained, its
thresholds re-picked as in section 8, and scored at the threshold of the source's rate
(as the detector applies it) on section 8's tests 1 and 3; the unseen-source rows keep
section 8's thresholds (the source's own for "seen", the pooled one for "unseen").

Little room is expected: #96's model flags 0.4 % (1 s) and 0.9 % (100 ms) of clean grid
windows, and catches 7 % (1 s) and 82 % (100 ms) of noise faults; a filter smooths
injected noise as well as healthy variation.

**Keep rule** (mean over the injection seeds, sample sd, on both grid sources): false
alarms below `none`'s mean minus max(its sd, 0.001), and every kind's recall at least
`none`'s mean minus max(its sd, 0.01). The generator and the unseen sources are
reported, not gated. Results: `research/cleaning/results.md`.

In [ ]:
# one setting per filter (#66 design); a filter replaces a sensor's raw series before
# the short-window columns are computed
CLEAN_WINDOW = 5  # readings in the centred moving median (odd: same length out)
EMA_ALPHA = 0.3  # weight of the newest reading in the exponential moving average
# the pipeline's Kalman (pipelines/denoise.py) with the setup tested here: forward
# filter, fixed noise, no jump pass-through, and repeated readings smoothed too
# (the rule that keeps them came later; these tables were built without it)
TESTED_KALMAN = KalmanSettings(keep_repeats=False)
KALMAN_PROCESS_RATIO = TESTED_KALMAN.process_ratio  # in the table cache key
CleanHandler = Callable[[FloatArray], FloatArray]


def filterable(values: FloatArray) -> bool:
    """Enough readings to filter, and not flat: otherwise every filter returns a
    copy (a stuck sensor must stay stuck)."""
    return len(values) >= PREPROCESS.min_samples and float(np.ptp(values)) > 0

In [ ]:
def no_filter(values: FloatArray) -> FloatArray:
    return values.copy()


def median_filter(values: FloatArray) -> FloatArray:
    """Centred moving median of CLEAN_WINDOW readings; the ends repeat their edge
    value."""
    if not filterable(values) or len(values) < CLEAN_WINDOW:
        return values.copy()
    padded = np.pad(values, CLEAN_WINDOW // 2, mode="edge")
    windows = np.lib.stride_tricks.sliding_window_view(padded, CLEAN_WINDOW)
    return np.median(windows, axis=1)


def ema_filter(values: FloatArray) -> FloatArray:
    """Exponential moving average, forward in time (a first-order low-pass)."""
    if not filterable(values):
        return values.copy()
    out = np.empty_like(values)
    out[0] = values[0]
    for index in range(1, len(values)):
        out[index] = EMA_ALPHA * values[index] + (1 - EMA_ALPHA) * out[index - 1]
    return out


def kalman_filter(values: FloatArray) -> FloatArray:
    """The pipeline's Kalman cleaning with the setup tested here
    (TESTED_KALMAN)."""
    return kalman_clean(CleanContext(values, TESTED_KALMAN, PREPROCESS.min_samples))


# filter name -> filter: one function per variant and a registry (lead, PR #97)
CLEAN_REGISTRY: dict[str, CleanHandler] = {
    "none": no_filter,
    "median": median_filter,
    "ema": ema_filter,
    "kalman": kalman_filter,
}


def tuned_kalman(values: FloatArray) -> FloatArray:
    """The pipeline's Kalman cleaning with the shipped setup (preprocess.kalman,
    section 11's pick)."""
    return kalman_clean(CleanContext(values, PREPROCESS.kalman, PREPROCESS.min_samples))


# the shipped setup's row; its tables are the default preprocess ones (clean=None),
# built and cached by section 11
TUNED_ROW = "kalman, tuned"
# row name -> the clean argument of its tables (None: the default preprocess)
CLEAN_ROWS: dict[str, str | None] = {name: name for name in CLEAN_REGISTRY} | {
    TUNED_ROW: None
}
# row name -> its filter, for the latency
ROW_FILTERS: dict[str, CleanHandler] = CLEAN_REGISTRY | {TUNED_ROW: tuned_kalman}

In [ ]:
# self-check (instant): every filter keeps the length, leaves short and flat series
# unchanged, and smooths a noisy one; kalman skips a zero noise estimate
_noisy = 50.0 + 0.01 * np.random.default_rng(0).standard_normal(200)
for _name, _filter in CLEAN_REGISTRY.items():
    _out = _filter(_noisy)
    assert len(_out) == len(_noisy) and np.isfinite(_out).all() and _out is not _noisy
    assert _name == "none" or np.diff(_out).std() < np.diff(_noisy).std(), _name
    for _same in (np.full(20, 0.3), np.arange(3.0)):
        assert np.array_equal(_filter(_same), _same), _name
assert list(CLEAN_REGISTRY) == ["none", "median", "ema", "kalman"]
# the tuned row is the pipeline's own default cleaning, on the default tables
assert list(CLEAN_ROWS) == [*CLEAN_REGISTRY, TUNED_ROW]
assert CLEAN_ROWS[TUNED_ROW] is None
assert np.array_equal(tuned_kalman(_noisy), PreProcess().clean(_noisy))
assert np.array_equal(CLEAN_REGISTRY["none"](_noisy), _noisy)
_steps = np.array([1.0, 1, 1, 1, 2, 2, 2, 2])  # median |step| is 0
assert np.array_equal(kalman_filter(_steps), _steps)

In [ ]:
# self-check (seconds): "none" gives the raw series, a filter changes only
# the candidate columns, and each filter has its own cache key
_window = [r.sensor for r in generator_readings(0, WINDOW_SECONDS)]
_plain = wide_window(PreProcess(), Decompose(), "none", _window, "none")
_cleaned = wide_window(PreProcess(), Decompose(), "none", _window, "median")
assert all(_cleaned[s][k] == _plain[s][k] for s in _plain for k in BASE_KEYS)
assert any(_cleaned[s][k] != _plain[s][k] for s in _plain for k in SHORT_MODEL_KEYS)
assert all(np.isfinite(list(_cleaned[s].values())).all() for s in _cleaned)
_keys = {name: table_cache_path(0, name) for name in CLEAN_REGISTRY}
assert len(set(_keys.values())) == 4
_alpha, EMA_ALPHA = EMA_ALPHA, 0.5
assert table_cache_path(0, "ema") != _keys["ema"]
assert table_cache_path(0, "none") == _keys["none"]
EMA_ALPHA = _alpha

In [ ]:
# self-check (seconds): the tables the model trains on read the cleaning live inference
# applies, and section 10's tables keep their cache keys when the default changes
_kalman_preprocess = PREPROCESS.model_copy(update={"clean_method": "kalman"})
_window = [r.sensor for r in generator_readings(0, WINDOW_SECONDS)]
_cleaning = PreProcess("none", settings=_kalman_preprocess)
_trained = wide_window(_cleaning, Decompose(), "none", _window)
_live = process_window(_cleaning, Decompose(), "none", _window)
assert all(_trained[s][k] == _live[s][k] for s in _live for k in SHORT_MODEL_KEYS)
_raw = wide_window(_cleaning, Decompose(), "none", _window, "none")
assert any(_raw[s][k] != _trained[s][k] for s in _live for k in SHORT_MODEL_KEYS)
# the cached section 10 tables (built before #66 added the cleaning settings); update
# this name, and rebuild, only when a preprocess or dataset setting really changes
assert table_cache_path(0, "none").name == "seed0_a1612717d319.pkl"
_names = {name: table_cache_path(0, name) for name in CLEAN_REGISTRY}
_default = table_cache_path(0)
# another default (no cleaning: the shipped one is kalman) names another table
_other = PREPROCESS.model_copy(update={"clean_method": "none"})
_preprocess, PREPROCESS = PREPROCESS, _other
assert {name: table_cache_path(0, name) for name in CLEAN_REGISTRY} == _names
assert table_cache_path(0) != _default
assert table_cache_path(0, preprocess=_other) == table_cache_path(0)
PREPROCESS = _preprocess

In [ ]:
RECALL_FLOOR = 0.01  # least recall drop that counts: several kinds score 1.000 ± 0.000
FALSE_ALARM_FLOOR = 0.001  # least false-alarm drop that counts (1 s rate: 0.004)
MODEL_KINDS = tuple(kind for kind in SHORT_KINDS if kind != "spoof")
# columns that describe a window's noise texture, for the false-alarm profile
NOISE_KEYS = (
    "n_haar_d1_energy",
    "sign_flip_share",
    "diff_autocorr",
    "rolling_variance_ratio",
    "max_abs_step",
)


def seed_mean_sd(rows: list[Metrics], metric: str) -> tuple[float, float]:
    """The metric's mean and sample sd over the seeds (sd 0 for one seed), as
    mean_sd computes them."""
    values = np.array([row[metric] for row in rows])
    return float(values.mean()), float(values.std(ddof=1)) if len(values) > 1 else 0.0


def keeps_cleaning(runs: Runs, clean: str) -> bool:
    """The #66 keep rule, ``clean`` against "none" on each grid source's test 1:
    false alarms drop by more than max(none's sd, FALSE_ALARM_FLOOR), and no kind's
    recall drops by more than max(none's sd, RECALL_FLOOR)."""
    for source in SHORT_SOURCES:
        table = runs[f"Test 1, {source}"]
        reference, sd = seed_mean_sd(table["none"], "false_alarms")
        mean, _ = seed_mean_sd(table[clean], "false_alarms")
        if mean >= reference - max(sd, FALSE_ALARM_FLOOR):
            return False
        for kind in MODEL_KINDS:
            reference, sd = seed_mean_sd(table["none"], f"recall_{kind}")
            mean, _ = seed_mean_sd(table[clean], f"recall_{kind}")
            if mean < reference - max(sd, RECALL_FLOOR):
                return False
    return True


def filter_rank(
    runs: Runs, latency: dict[str, Metrics], clean: str
) -> tuple[float, float]:
    """Mean false alarms over both grid sources' test 1, then the filter's p50 time:
    lower is better."""
    alarms = [
        seed_mean_sd(runs[f"Test 1, {source}"][clean], "false_alarms")[0]
        for source in SHORT_SOURCES
    ]
    return float(np.mean(alarms)), latency[clean]["p50_ms"]


def pick_filter(runs: Runs, latency: dict[str, Metrics]) -> str | None:
    """The passing filter with the fewest false alarms (ties: the faster), or None."""
    passing = [
        name for name in CLEAN_REGISTRY if name != "none" and keeps_cleaning(runs, name)
    ]
    if not passing:
        return None
    return min(passing, key=partial(filter_rank, runs, latency))


# test 1 scored again at the threshold that gives the baseline's false alarms
MATCHED_TITLE = "Test 1, {}, at the baseline's false alarms"


def threshold_for_alarms(normal_scores: FloatArray, share: float) -> float:
    """The threshold above which at most ``share`` of these normal windows score, so
    filters are compared at one false-alarm rate (their own re-picked thresholds
    flag different shares)."""
    allowed = int(share * len(normal_scores))
    ranked = np.sort(normal_scores)[::-1]
    return float(ranked[allowed]) if allowed < len(ranked) else -math.inf

In [ ]:
# self-check (instant): the keep rule's tolerances and floors, and the pick
def _runs(fa: float, noise: float, noise_sd: float = 0.0) -> list[Metrics]:
    """Three seeds' metrics: false alarms fa (sd 0), noise recall with sample sd
    noise_sd, every other kind 0.9."""
    return [
        {"false_alarms": fa}
        | {f"recall_{k}": 0.9 for k in MODEL_KINDS}
        | {"recall_noise": noise + d}
        for d in (-noise_sd, 0.0, noise_sd)
    ]


def _both(rows: dict[str, list[Metrics]]) -> Runs:
    """The same rows for both grid sources' test 1."""
    return {f"Test 1, {source}": rows for source in SHORT_SOURCES}


_none = _runs(0.004, 0.80)
assert not keeps_cleaning(_both({"none": _none, "x": _runs(0.004, 0.80)}), "x")
assert keeps_cleaning(_both({"none": _none, "x": _runs(0.002, 0.80)}), "x")
# a drop within the 0.001 floor is seed noise
assert not keeps_cleaning(_both({"none": _none, "x": _runs(0.0035, 0.80)}), "x")
assert not keeps_cleaning(_both({"none": _none, "x": _runs(0.002, 0.78)}), "x")
assert keeps_cleaning(_both({"none": _none, "x": _runs(0.002, 0.795)}), "x")
# within one of none's seed sds (0.02) on noise recall
_wide = {"none": _runs(0.004, 0.80, 0.02), "x": _runs(0.002, 0.785)}
assert keeps_cleaning(_both(_wide), "x")
# one grid source failing is enough
_mixed = {
    f"Test 1, {SHORT_SOURCES[0]}": {"none": _none, "x": _runs(0.002, 0.80)},
    f"Test 1, {SHORT_SOURCES[1]}": {"none": _none, "x": _runs(0.004, 0.80)},
}
assert not keeps_cleaning(_mixed, "x")
_latency = {name: {"p50_ms": 1.0, "p99_ms": 1.0} for name in CLEAN_REGISTRY}
_all = {"none": _none, "median": _runs(0.002, 0.80), "ema": _runs(0.001, 0.80)}
_all["kalman"] = _none
assert pick_filter(_both(_all), _latency) == "ema"
_all["ema"] = _all["median"]  # a tie: the faster filter
assert pick_filter(_both(_all), _latency | {"median": {"p50_ms": 2.0}}) == "ema"
assert pick_filter(_both(dict.fromkeys(CLEAN_REGISTRY, _none)), _latency) is None

In [ ]:
# self-check (instant): a threshold that flags at most a share of the normal windows
_normal = np.arange(1, 10) / 10  # 0.1 ... 0.9
assert (_normal > threshold_for_alarms(_normal, 0.25)).sum() == 2  # 2 of 9 allowed
assert (_normal > threshold_for_alarms(_normal, 0.0)).sum() == 0
assert (_normal > threshold_for_alarms(_normal, 1.0)).sum() == 9
_tied = np.array([0.5, 0.5, 0.5, 0.2])
assert (_tied > threshold_for_alarms(_tied, 0.5)).sum() <= 2  # ties never exceed it

In [ ]:
def clean_seed(
    injection_seed: int, clean: str | None, alarms: dict[str, float] | None = None
) -> tuple[Runs, BaselineModel, ShortTables]:
    """Section 8's short-window model with the filter ``clean`` (None: the default
    preprocess cleaning, rows named TUNED_ROW): fitted and its
    thresholds picked as short_seed does, scored at the threshold of each test
    source's rate (tests 1 and 3, rows named ``clean``), and with each source held out
    of training ("Unseen source"). ``alarms``: per grid source, a false-alarm share
    test 1 is also scored at (rows under MATCHED_TITLE). Also the model and its
    tables."""
    tables = select_tables(cached_short_tables(injection_seed, clean), SHORT_MODEL_KEYS)
    name = TUNED_ROW if clean is None else clean
    data = tables.train
    train = short_train(data)
    short, thresholds, _ = short_fit(data, train, keys=SHORT_MODEL_KEYS)
    runs: Runs = {}
    for title, (d, rows) in test_rows(tables).items():
        source = title.split(", ")[1]  # "Test 1, grid_frequency" -> grid_frequency
        threshold = short.rate_thresholds.get(SOURCE_RATE[source], short.threshold)
        add_run(runs, title, name, short_report(short, d, rows, threshold))
    tests = test_rows(tables)
    for source, share in (alarms or {}).items():
        d, rows = tests[f"Test 1, {source}"]
        normal = rows & (class_index(d, CLASSES) == 0)
        threshold = threshold_for_alarms(short.score(d.features[normal]), share)
        report = short_report(short, d, rows, threshold)
        add_run(runs, MATCHED_TITLE.format(source), name, report)
    for held_out, metrics in unseen_source_reports(
        tables, train, short, thresholds
    ).items():
        add_run(runs, "Unseen source", f"{name} / {held_out}", metrics)
    return runs, short, tables


def false_alarm_profile(
    model: BaselineModel, tables: ShortTables
) -> dict[str, Metrics]:
    """What the model's false alarms look like: per grid source, the median of each
    noise-texture column (NOISE_KEYS) on the clean test windows it flags at the
    threshold of the source's rate, on those it does not flag, and on the injected
    noise windows. If the flagged clean windows are not the noisier ones, a filter has
    nothing to remove there."""
    data = tables.train
    test = ~data.train & short_usable(data)
    columns = [model.keys.index(key) for key in NOISE_KEYS]
    profile = {}
    for source in SHORT_SOURCES:
        rows = test & (data.dataset == source)
        threshold = model.rate_thresholds.get(SOURCE_RATE[source], model.threshold)
        flagged = np.zeros(len(rows), dtype=bool)
        flagged[rows] = model.score(data.features[rows]) > threshold
        clean = rows & (data.kind == "")
        groups = {
            "flagged clean": clean & flagged,
            "unflagged clean": clean & ~flagged,
            "injected noise": rows & (data.kind == "noise"),
        }
        for name, chosen in groups.items():
            values = data.features[chosen][:, columns]
            medians = (
                np.median(values, axis=0)
                if chosen.any()
                else np.full(len(columns), np.nan)
            )
            profile[f"{source}: {name}"] = {"windows": float(chosen.sum())} | dict(
                zip(NOISE_KEYS, medians.tolist(), strict=True)
            )
    return profile


def filter_latency(clean: str) -> Metrics:
    """The filter's own time on one live-sized window (latency_window: TWIN_SENSORS
    sensors x 200 readings); every sensor's raw series is built before the clock
    starts."""
    preprocessor = PreProcess()
    unique, _ = preprocessor.deduplicate(preprocessor.finite(latency_window()))
    by_sensor: dict[str, list[Sensor]] = {}
    for reading in unique:
        by_sensor.setdefault(reading.sensor_id, []).append(reading)
    series = [
        preprocessor.sensor_series(readings)[1] for readings in by_sensor.values()
    ]
    handler = ROW_FILTERS[clean]
    seconds = []
    for run in range(WARMUP + REPEATS):
        started = perf_counter()
        for raw in series:
            handler(raw)
        if run >= WARMUP:
            seconds.append(perf_counter() - started)
    return {
        "p50_ms": 1000 * float(np.percentile(seconds, 50)),
        "p99_ms": 1000 * float(np.percentile(seconds, 99)),
    }

In [ ]:
def clean_main() -> None:
    """Every filter on every injection seed (test 1 also at the baseline's false
    alarms), the baseline's false-alarm profile, the
    filters' latency, the keep rule and the pick; prints the tables (mean ± sd over
    the seeds). Saves no model."""
    started = perf_counter()
    runs: Runs = {}
    for injection_seed in INJECTION_SEEDS:
        alarms: dict[str, float] = {}  # the baseline's test 1 false alarms, this seed
        for row, clean in CLEAN_ROWS.items():
            seed_runs, model, tables = clean_seed(injection_seed, clean, alarms)
            merge_runs(runs, seed_runs)
            if clean == "none":
                for source in SHORT_SOURCES:
                    baseline = seed_runs[f"Test 1, {source}"]["none"][0]
                    alarms[source] = baseline["false_alarms"]
                    add_run(runs, MATCHED_TITLE.format(source), "none", baseline)
                for name, metrics in false_alarm_profile(model, tables).items():
                    add_run(runs, "False alarms of the baseline", name, metrics)
            print(f"{row}, injection seed {injection_seed} done", flush=True)
    latency = {row: filter_latency(row) for row in CLEAN_ROWS}
    settings = {
        "filters": list(CLEAN_ROWS),
        "tuned_kalman": PREPROCESS.kalman.model_dump(),
        "clean_window": CLEAN_WINDOW,
        "ema_alpha": EMA_ALPHA,
        "kalman_process_ratio": KALMAN_PROCESS_RATIO,
        "injection_seeds": INJECTION_SEEDS,
        "recall_floor": RECALL_FLOOR,
        "false_alarm_floor": FALSE_ALARM_FLOOR,
        "seconds": round(perf_counter() - started),
    }
    print(f"settings: {json.dumps(settings)}")
    print(
        f"filter latency, {TWIN_SENSORS} sensors x 200 readings: {json.dumps(latency)}"
    )
    for row in CLEAN_ROWS:
        if row != "none":
            print(f"{row}: keeps={keeps_cleaning(runs, row)}")
    print(f"picked: {pick_filter(runs, latency) or 'none'}")
    for title, table in runs.items():
        print(f"\n## {title}\n")
        print(markdown_table({name: mean_sd(rows) for name, rows in table.items()}))

Long-running: for each injection seed, the four tables (built once and cached, about
10 minutes each; the `kalman, tuned` row reads section 11's cached tables), then a
model fit and three held-out-source fits per row. Needs
section 8's cells.

In [ ]:
clean_main()

## 11. Kalman tuning

The lead decided on PR #114 that the readings are cleaned before both models ("we need
noise cleaning"), and the Kalman filter was chosen over EMA and median (section 10:
median erases spikes; Kalman keeps more stuck faults than EMA at the same PR-AUC).
This section tunes the short-window model's setup before it ships: forward filter or
smoother (which also uses the readings after each point), the noise fixed as a ratio
or estimated from the window, jumps passed through or smoothed, and the smoothing
strength (`pipelines/denoise.py`, `KalmanSettings`). Runs of repeated readings always
come out as they are (`keep_repeats`), so a stuck value stays flat.

Each setup's section 8 tables are built and scored on validation windows only
(section 8's `score_variant`): the pick is the setup with the highest mean PR-AUC on
sources the model never trained on (each of the three held out in turn, the generator
included). Cleaning ships whatever the comparison with no cleaning shows, so there is
no false-alarm gate: each source's validation false alarms, at the threshold the
detector would apply live (the source's rate), are reported. A `no cleaning` row,
scored the same way on section 10's raw tables, is printed next to stage A and the
confirmation. Stages on injection seed 0, the best three confirmed on seeds 0-2. The
pick becomes the default in `config/settings.yaml`; section 8 then retrains the saved
model and section 10 reports it next to no filter. Results:
`research/cleaning/results.md`.

In [ ]:
EDGE_STEPS = 2  # most extra grid steps past each end of a grid
TUNE_SEED = 0  # injection seed of the staged search; the best three get all seeds
STAGE_RATIOS = (0.03, 0.1, 0.3, 1.0)  # fixed noise: process / measurement
STAGE_SCALES = (0.5, 1.0, 2.0)  # estimated noise: process noise multiplied by
STAGE_JUMPS = (2.0, 3.0, 4.0)  # prediction sds past which a reading passes through
ScoredSetups = dict[str, tuple[KalmanSettings, list[Metrics]]]


def kalman_preprocess(setup: KalmanSettings) -> PreprocessSettings:
    """The default preprocess settings with this Kalman cleaning switched on."""
    return PREPROCESS.model_copy(update={"clean_method": "kalman", "kalman": setup})


def setup_name(setup: KalmanSettings) -> str:
    """A readable name: mode, noise and its strength, jumps."""
    strength = (
        f"ratio {setup.process_ratio:g}"
        if setup.noise == "fixed"
        else f"scale {setup.process_scale:g}"
    )
    jumps = "no jumps" if setup.jump_sigmas is None else f"jumps {setup.jump_sigmas:g}"
    return f"{setup.mode}, {setup.noise} {strength}, {jumps}"


def simplicity(setup: KalmanSettings) -> tuple[int, int, bool]:
    """Lower is simpler: filter before smoother, fixed before estimated noise, no
    pass-through before pass-through."""
    return (
        KALMAN_MODES.index(setup.mode),
        KALMAN_NOISES.index(setup.noise),
        setup.jump_sigmas is not None,
    )


def unseen_score(scored: ScoredSetups, name: str) -> float:
    return mean_metrics(scored[name][1])["unseen_mean"]


def pick_rank(scored: ScoredSetups, name: str) -> tuple[float, tuple[int, int, bool]]:
    """Lower ranks first: the higher unseen PR-AUC, then (an exact tie) the simpler."""
    return -unseen_score(scored, name), simplicity(scored[name][0])


def pick_setup(scored: ScoredSetups) -> str:
    """The setup with the highest mean unseen-source PR-AUC. No false-alarm gate:
    cleaning ships either way (the user's call), the alarms are reported."""
    return min(scored, key=partial(pick_rank, scored))


def ratio_step(value: float, direction: int) -> float:
    # divide for the lower step: 0.03 / 3 is exactly 0.01, 0.03 * 3**-1 is not
    return value * 3.0 if direction > 0 else value / 3.0


def scale_step(value: float, direction: int) -> float:
    return value * 2.0 if direction > 0 else value / 2.0


def jump_step(value: float, direction: int) -> float:
    return value + direction


def extend_grid(
    base: tuple[float, ...],
    tried: list[float],
    picked: float,
    step: Callable[[float, int], float],
    limit: int = EDGE_STEPS,
) -> float | None:
    """The next value to try when the pick sits on an end of the values tried, at
    most ``limit`` past each end of the base grid; None when it is inside (or the
    next value would not be positive)."""
    beyond_low = sum(value < min(base) for value in tried)
    beyond_high = sum(value > max(base) for value in tried)
    if picked == min(tried) and beyond_low < limit:
        candidate = step(picked, -1)
    elif picked == max(tried) and beyond_high < limit:
        candidate = step(picked, 1)
    else:
        return None
    return candidate if candidate > 0 else None

In [ ]:
def validation_alarms(
    model: BaselineModel, data: WindowDataset, train: BoolArray
) -> dict[str, float]:
    """Per source, the share of its clean validation windows flagged at the threshold
    the detector would apply live (the source's sampling rate: the generator gets the
    1 s grid's), as alarms_<source>."""
    validation = late_windows(data, train) & (data.kind == "")
    alarms = {}
    for source in np.unique(data.dataset[validation]):
        rows = validation & (data.dataset == source)
        threshold = model.rate_thresholds.get(SOURCE_RATE[str(source)], model.threshold)
        flagged = model.score(data.features[rows]) > threshold
        alarms[f"alarms_{source}"] = float(flagged.mean())
    return alarms


def score_tables(wide: ShortTables) -> Metrics:
    """One seed's tables on validation windows: section 8's score_variant (in-source
    and unseen PR-AUC per source, unseen_mean) and the false alarms of the model
    short_fit trains (one extra fit: score_variant keeps its models)."""
    metrics = score_variant(wide, SHORT_MODEL_KEYS)
    data = select(wide.train, SHORT_MODEL_KEYS)
    train = short_train(data)
    model, _, _ = short_fit(data, train, keys=SHORT_MODEL_KEYS)
    return metrics | validation_alarms(model, data, train)


def tune_score(injection_seed: int, setup: KalmanSettings) -> Metrics:
    """One setup on one seed (its tables built once, cached)."""
    return score_tables(
        cached_short_tables(injection_seed, preprocess=kalman_preprocess(setup))
    )


def baseline_score(injection_seed: int) -> Metrics:
    """No cleaning, scored the same way, on section 10's raw tables."""
    return score_tables(cached_short_tables(injection_seed, "none"))


def with_ratio(setup: KalmanSettings, value: float) -> KalmanSettings:
    return setup.model_copy(update={"noise": "fixed", "process_ratio": value})


def with_scale(setup: KalmanSettings, value: float) -> KalmanSettings:
    return setup.model_copy(update={"noise": "estimated", "process_scale": value})


def with_jumps(setup: KalmanSettings, value: float) -> KalmanSettings:
    return setup.model_copy(update={"jump_sigmas": value})

In [ ]:
# self-check (instant): the pick, the grid edges, the settings copy
def _scored(unseen: float, alarm: float) -> list[Metrics]:
    return [{"unseen_mean": unseen, "alarms_grid_frequency": alarm}]


_filter, _smoother = KalmanSettings(), KalmanSettings(mode="smoother")
_runs = {
    "loud": (_smoother.model_copy(update={"process_ratio": 0.3}), _scored(0.9, 0.05)),
    "smoother": (_smoother, _scored(0.704, 0.01)),
    "filter": (_filter, _scored(0.700, 0.01)),
}
assert pick_setup(_runs) == "loud"  # no false-alarm gate: the best unseen score wins
_tie = {
    "smoother": (_smoother, _scored(0.7, 0.01)),
    "filter": (_filter, _scored(0.7, 0.02)),
}
assert pick_setup(_tie) == "filter"  # an exact tie: the simpler one
_ratios = (0.03, 0.1, 0.3, 1.0)
assert extend_grid(_ratios, list(_ratios), 0.1, ratio_step) is None  # interior
assert extend_grid(_ratios, list(_ratios), 0.03, ratio_step) == 0.01
assert extend_grid(_ratios, [*_ratios, 0.01, 0.01 / 3], 0.01 / 3, ratio_step) is None
assert extend_grid(_ratios, [*_ratios, 3.0], 3.0, ratio_step, limit=1) is None
assert extend_grid(_ratios, list(_ratios), 1.0, ratio_step, limit=1) == 3.0
assert extend_grid((2.0, 3.0, 4.0), [2.0, 3.0, 4.0, 1.0], 1.0, jump_step) is None
_tuned = kalman_preprocess(_smoother)
assert _tuned.clean_method == "kalman" and _tuned.kalman == _smoother
assert _tuned.model_dump(exclude={"clean_method", "kalman"}) == PREPROCESS.model_dump(
    exclude={"clean_method", "kalman"}
)

In [ ]:
BASELINE_NAME = "no cleaning"


def run_stage(
    title: str,
    setups: list[KalmanSettings],
    seeds: tuple[int, ...],
    scored: ScoredSetups,
    baseline: list[Metrics] | None = None,
) -> list[str]:
    """Score each setup on these seeds (a setup already scored on them is reused),
    print the stage's table (mean ± sd; ``baseline``, the no-cleaning rows of these
    seeds, first when given) and return the setups' names."""
    names = []
    for setup in setups:
        name = setup_name(setup)
        done = scored.get(name, (setup, []))[1]
        missing = seeds[len(done) :]
        scored[name] = (setup, [*done, *(tune_score(s, setup) for s in missing)])
        names.append(name)
        print(f"{title}: {name} done", flush=True)
    rows = {} if baseline is None else {BASELINE_NAME: mean_sd(baseline)}
    rows |= {n: mean_sd(scored[n][1][: len(seeds)]) for n in names}
    print(f"\n## {title}\n")
    print(markdown_table(rows))
    return names


def extend_stage(
    title: str,
    names: list[str],
    tried: list[float],
    base: tuple[float, ...],
    make: Callable[[float], KalmanSettings],
    step: Callable[[float, int], float],
    scored: ScoredSetups,
) -> list[str]:
    """While the best of these setups sits on an end of its grid, try the next value
    past it (extend_grid), up to EDGE_STEPS each way."""
    names, tried = list(names), list(tried)
    while True:
        picked = pick_setup({n: scored[n] for n in names})
        value = extend_grid(base, tried, tried[names.index(picked)], step)
        if value is None:
            return names
        names += run_stage(
            f"{title}, past the edge", [make(value)], (TUNE_SEED,), scored
        )
        tried.append(value)


def confirm(names: list[str], scored: ScoredSetups, baseline: list[Metrics]) -> str:
    """The best three on seed 0, scored on every injection seed; the pick on their
    means."""
    top = sorted(names, key=partial(pick_rank, scored))[:3]
    title = "Confirmed on all seeds"
    run_stage(title, [scored[n][0] for n in top], INJECTION_SEEDS, scored, baseline)
    return pick_setup(
        {n: (scored[n][0], scored[n][1][: len(INJECTION_SEEDS)]) for n in top}
    )


def kalman_main() -> None:
    """The staged search: A mode x jumps, B the noise and its strength on A's best,
    C the jump threshold if jumps won; edges extended; the best three confirmed on all
    seeds. Prints every stage (no cleaning beside A and the confirmation), the pick and
    its settings."""
    started, scored = perf_counter(), {}
    seed = (TUNE_SEED,)
    baseline = [baseline_score(s) for s in INJECTION_SEEDS]  # cached raw tables
    stage_a = [
        KalmanSettings(mode=m, jump_sigmas=j) for m in KALMAN_MODES for j in (None, 3.0)
    ]
    names = run_stage(
        "Stage A: filter or smoother, jumps", stage_a, seed, scored, baseline[:1]
    )
    base = scored[pick_setup({n: scored[n] for n in names})][0]
    fixed = run_stage(
        "Stage B: fixed noise",
        [with_ratio(base, r) for r in STAGE_RATIOS],
        seed,
        scored,
    )
    fixed = extend_stage(
        "Stage B: fixed noise",
        fixed,
        list(STAGE_RATIOS),
        STAGE_RATIOS,
        partial(with_ratio, base),
        ratio_step,
        scored,
    )
    estimated = run_stage(
        "Stage B: estimated noise",
        [with_scale(base, s) for s in STAGE_SCALES],
        seed,
        scored,
    )
    estimated = extend_stage(
        "Stage B: estimated noise",
        estimated,
        list(STAGE_SCALES),
        STAGE_SCALES,
        partial(with_scale, base),
        scale_step,
        scored,
    )
    best = scored[pick_setup({n: scored[n] for n in [*names, *fixed, *estimated]})][0]
    if best.jump_sigmas is not None:
        jumps = run_stage(
            "Stage C: jump threshold",
            [with_jumps(best, j) for j in STAGE_JUMPS],
            seed,
            scored,
        )
        extend_stage(
            "Stage C: jump threshold",
            jumps,
            list(STAGE_JUMPS),
            STAGE_JUMPS,
            partial(with_jumps, best),
            jump_step,
            scored,
        )
    picked = confirm(list(scored), scored, baseline)
    print(f"\nseconds: {round(perf_counter() - started)}")
    print(f"picked: {picked}")
    print(f"settings: {json.dumps(scored[picked][0].model_dump())}")

Long-running: each new setup builds its section 8 tables once (cached, about 10-20
minutes per seed), then five model fits per seed; about 15-20 setups on seed 0 and
three on seeds 1-2. The no-cleaning rows read section 10's cached raw tables.

In [ ]:
kalman_main()

## 12. Long-window Kalman tuning

The lead asked for both models to be cleaned (PR #114). The long-window model cleans
each sensor's 5-minute bucket grid (`LongPreProcess.clean`, before every feature)
with its own setup (`long_window.kalman`): a bucket mean holds far less noise than one
reading, so section 11's pick need not be the best here. One table build takes about
50 minutes, so the search is small and starts from section 11's pick (the
`preprocess.kalman` default once it is set): the pick itself, its smoothing strength
one step weaker and one step stronger (one more step if an end wins), and the pick
with the other mode. Scored like section 11, on validation windows only (section 9's
late rows of the fit part), injection seed 0, 24 h windows: the pick is the highest
mean PR-AUC on sources the model never trained on (each source with both classes in
its validation rows held out in turn). The pooled validation PR-AUC, false alarms per
sensor-day and stuck and drift recall of the model fitted on every source are
reported, next to a `no cleaning` row. The pick becomes `long_window.kalman`; section 9
then retrains the saved model. Results: `research/cleaning/results.md`.

In [ ]:
LONG_TUNE_SEED = 0  # one seed only: one table build takes about 50 minutes
LONG_TUNE_WINDOW = LONG_WINDOW.window_seconds[0]
LONG_EDGE_STEPS = 1  # most extra strength steps past an end of the long search
# noise mode -> the setting that holds its smoothing strength, and its grid step
STRENGTHS: dict[str, tuple[str, Callable[[float, int], float]]] = {
    "fixed": ("process_ratio", ratio_step),
    "estimated": ("process_scale", scale_step),
}
OTHER_MODE = {"filter": "smoother", "smoother": "filter"}


def long_settings(setup: KalmanSettings | None) -> LongWindowSettings:
    """The long-window settings with this Kalman cleaning; None: no cleaning."""
    if setup is None:
        return LONG_WINDOW.model_copy(update={"clean_method": "none"})
    return LONG_WINDOW.model_copy(update={"clean_method": "kalman", "kalman": setup})


def strength_of(setup: KalmanSettings) -> float:
    return getattr(setup, STRENGTHS[setup.noise][0])


def with_strength(setup: KalmanSettings, value: float) -> KalmanSettings:
    # 6 significant digits: 0.3 * 3 is 0.9000000000000001, and the pick goes to
    # settings.yaml as it prints
    return setup.model_copy(update={STRENGTHS[setup.noise][0]: float(f"{value:.6g}")})


def long_setups(pick: KalmanSettings) -> list[KalmanSettings]:
    """Section 11's pick, its strength one step down and one step up, and the pick
    with the other mode."""
    step, value = STRENGTHS[pick.noise][1], strength_of(pick)
    return [
        pick,
        with_strength(pick, step(value, -1)),
        with_strength(pick, step(value, 1)),
        pick.model_copy(update={"mode": OTHER_MODE[pick.mode]}),
    ]


def long_unseen(
    data: WindowDataset, early: BoolArray, late: BoolArray, truth: IntArray
) -> dict[str, float]:
    """Test 4 on validation rows: per source, the PR-AUC on its validation rows of a
    model fitted on the other sources, as unseen_<source> (sources with both classes
    there); unseen_mean over them."""
    columns = [LONG_COLUMNS.index(key) for key in LONG_KEYS]
    scores = {}
    for source in np.unique(data.dataset):
        own = data.dataset == source
        rows = late & own
        positive = truth[rows] > 0
        if positive.all() or not positive.any() or not (early & ~own).any():
            continue
        model, _ = long_fit(
            data, early & ~own, late & ~own, LONG_KEYS, LONG_TUNE_WINDOW
        )
        predicted = model.score(data.features[rows][:, columns])
        scores[f"unseen_{source}"] = float(average_precision_score(positive, predicted))
    return scores | {"unseen_mean": float(np.mean(list(scores.values())))}


def long_tune_score(setup: KalmanSettings | None) -> Metrics:
    """One long setup (None: no cleaning) on seed 0's validation rows: the model
    fitted on every source (PR-AUC, false alarms per sensor-day at its budget
    threshold, stuck and drift recall) and the unseen sources."""
    data = long_table(
        LONG_TUNE_WINDOW, LONG_TUNE_SEED, LONG_SOURCES, long_settings(setup)
    )
    early, late, _ = long_splits(data)
    truth = class_index(data, CLASSES)
    model, _ = long_fit(data, early, late, LONG_KEYS, LONG_TUNE_WINDOW)
    scores, flagged = long_scores(model, data, late)
    report = long_report(
        truth[late], data.kind[late], scores, flagged, LONG_TUNE_WINDOW
    )
    kept = ("pr_auc", "false_alarms_per_sensor_day", "recall_stuck", "recall_drift")
    return {f"val_{key}": report[key] for key in kept} | long_unseen(
        data, early, late, truth
    )

In [ ]:
# self-check (instant): the long settings, their cache keys, the setups around a pick
_w = LONG_TUNE_WINDOW
_path = partial(long_table_path, _w, 0, LONG_SOURCES)
assert long_table_path(_w, 0, LONG_SOURCES) == _path(LONG_WINDOW)
assert long_settings(None).clean_method == "none"
_long = long_settings(KalmanSettings())
assert _long.clean_method == "kalman" and _path(_long) != _path(long_settings(None))
# the retrain reads the pick back from settings.yaml: a validated copy must name the
# same cached table
assert _path(LongWindowSettings.model_validate(_long.model_dump())) == _path(_long)
_fixed = long_setups(KalmanSettings(mode="smoother", process_ratio=0.3))
assert [s.process_ratio for s in _fixed[:3]] == [0.3, 0.1, 0.9]
assert _fixed[3].mode == "filter" and _fixed[3].process_ratio == 0.3
_estimated = long_setups(KalmanSettings(noise="estimated"))
assert [s.process_scale for s in _estimated[:3]] == [1.0, 0.5, 2.0]
assert _estimated[3].mode == "smoother" and _estimated[3].noise == "estimated"
assert strength_of(with_strength(_fixed[0], 0.05)) == 0.05

In [ ]:
def long_score_setups(setups: list[KalmanSettings], scored: ScoredSetups) -> list[str]:
    """Score the setups not scored yet; their names."""
    names = []
    for setup in setups:
        name = setup_name(setup)
        if name not in scored:
            scored[name] = (setup, [long_tune_score(setup)])
            print(f"long: {name} done", flush=True)
        names.append(name)
    return names


def long_kalman_main(pick: KalmanSettings | None = None) -> None:
    """The long search around section 11's pick (default: the preprocess.kalman
    setting); prints the table (no cleaning first), the pick and its settings."""
    started = perf_counter()
    pick = get_settings().preprocess.kalman if pick is None else pick
    baseline = long_tune_score(None)
    print("long: no cleaning done", flush=True)
    scored: ScoredSetups = {}
    setups = long_setups(pick)
    names = long_score_setups(setups, scored)
    # the strength grid: the pick and its two steps (the other mode stands apart)
    grid = tuple(sorted(strength_of(setup) for setup in setups[:3]))
    best = scored[pick_setup({n: scored[n] for n in names[:3]})][0]
    step = STRENGTHS[pick.noise][1]
    value = extend_grid(grid, list(grid), strength_of(best), step, LONG_EDGE_STEPS)
    if value is not None:
        long_score_setups([with_strength(pick, value)], scored)
    picked = pick_setup(scored)
    rows = {BASELINE_NAME: mean_sd([baseline])}
    rows |= {name: mean_sd(runs) for name, (_, runs) in scored.items()}
    print("\n## Long-window Kalman tuning, seed 0, validation rows\n")
    print(markdown_table(rows))
    print(f"\nseconds: {round(perf_counter() - started)}")
    print(f"picked: {picked}")
    print(f"settings: {json.dumps(scored[picked][0].model_dump())}")

Long-running: one table build per setup (about 50 minutes each with no long cache),
five or six setups and the no-cleaning row, then 20 model fits per setup.

In [ ]:
long_kalman_main()